# Pipeline RGB monocular + YOLO ProFSAM + Arduino + rotas por borda/centróides — v8

Versão sem treinamento. Usa modelo pronto para fogo/chama, detector COCO para humanos, calibração de servos por 3 pontos, entrada por câmera ou vídeo da pasta e planejamento de combate em duas estratégias:

1. **Chamas grandes**: rota por pontos gerados ao longo da borda inferior da caixa da chama, com offset configurável.
2. **Chamas pequenas/remanescentes**: rota conectando centróides das regiões detectadas.

A mira enviada aos servos considera uma correção de queda do jato: a mira fica acima do ponto de impacto estimado.

In [1]:
# ============================================================
# CÉLULA 1 - Imports, caminhos e estrutura de pastas
# RGB monocular: sem calibração estéreo, sem profundidade e sem paletas térmicas
# Versão sem treinamento: usa modelo YOLO pronto ProFSAM para chama/fogo.
# ============================================================

from pathlib import Path
import os
import sys
import json
import time
import math
import glob
import shutil
import subprocess
import warnings
import urllib.request
from datetime import datetime

import numpy as np
import cv2

from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# Localização da raiz do projeto BLAZE
# ------------------------------------------------------------
def localizar_raiz_blaze(start=None):
    """
    Procura a pasta BLAZE subindo a partir do diretório atual.
    Se o notebook estiver fora do repositório, usa um caminho comum do projeto
    ou, por último, o diretório atual.
    """
    start = Path(start or Path.cwd()).resolve()

    for p in [start, *start.parents]:
        if p.name == "BLAZE":
            return p
        if (p / "vision").exists() and (p / "vision" / "fire_detect").exists():
            return p

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto BLAZE. "
        "Abra o notebook a partir de uma pasta dentro do repositório."
    )

PROJECT_ROOT = localizar_raiz_blaze()
BASE_DIR = PROJECT_ROOT / "vision" / "fire_detect" / "fire_monitor_RGB_monocular"
os.makedirs(BASE_DIR, exist_ok=True)
os.chdir(BASE_DIR)

# ------------------------------------------------------------
# Pastas principais do módulo monocular RGB
# ------------------------------------------------------------
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
RUNS_DIR = BASE_DIR / "runs"
ARDUINO_DIR = BASE_DIR / "arduino"
CAPTURES_DIR = BASE_DIR / "captures"
OUTPUTS_DIR = BASE_DIR / "outputs"
PARAMS_DIR = BASE_DIR / "parameters_setups"
OFFICIAL_SETUPS_DIR = PARAMS_DIR / "official"
USER_SETUPS_DIR = PARAMS_DIR / "user"

for d in [
    DATA_DIR, MODELS_DIR, RUNS_DIR, ARDUINO_DIR, CAPTURES_DIR, OUTPUTS_DIR,
    PARAMS_DIR, OFFICIAL_SETUPS_DIR, USER_SETUPS_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

FIRE_SETUPS_FILE = USER_SETUPS_DIR / "rgb_monocular_fire_monitor_setups.json"
DEFAULT_FIRE_MODEL_PATH = MODELS_DIR / "fire_yolo_best.pt"
DEFAULT_PERSON_MODEL_NAME = "yolo11n.pt"  # modelo COCO usado para detectar humanos/person

# ------------------------------------------------------------
# Parâmetros gerais de câmera
# ------------------------------------------------------------
CAM_INDEX = 0
CAP_WIDTH = 640
CAP_HEIGHT = 480
CAP_FPS = 30

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_DIR:", BASE_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("FIRE_SETUPS_FILE:", FIRE_SETUPS_FILE)
print("DEFAULT_FIRE_MODEL_PATH:", DEFAULT_FIRE_MODEL_PATH)
print("\nObservação: este notebook usa apenas uma câmera RGB monocular e não realiza treinamento.")
print("Modelo padrão de chama/fogo: UEmmanuel5/ProFSAM-Fire-Detector / Fire_best.pt")


PROJECT_ROOT: .
BASE_DIR: vision/fire_detect/fire_monitor_RGB_monocular
MODELS_DIR: vision/fire_detect/fire_monitor_RGB_monocular/models
FIRE_SETUPS_FILE: vision/fire_detect/fire_monitor_RGB_monocular/parameters_setups/user/rgb_monocular_fire_monitor_setups.json
DEFAULT_FIRE_MODEL_PATH: vision/fire_detect/fire_monitor_RGB_monocular/models/fire_yolo_best.pt

Observação: este notebook usa apenas uma câmera RGB monocular e não realiza treinamento.
Modelo padrão de chama/fogo: UEmmanuel5/ProFSAM-Fire-Detector / Fire_best.pt


In [ ]:
# ============================================================
# CÉLULA 2 - Modelo YOLO pronto para chama/fogo
# ============================================================
# Esta versão NÃO treina modelo e NÃO usa dataset.
# Ela baixa ou registra um arquivo .pt já treinado para detectar chama/fogo.
#
# Modelo padrão sugerido:
# - UEmmanuel5/ProFSAM-Fire-Detector, no Hugging Face.
# - Peso esperado: Fire_best.pt.
# - Modelo fire-only; o notebook usa a detecção de chama/fogo para acionar a trajetória.
# ============================================================

import os
import sys
import shutil
import subprocess
import urllib.request
from pathlib import Path
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

FIRE_MODEL_READY = None

PRETRAINED_FIRE_MODELS = {
    "YOLO11n Fire — Hugging Face UEmmanuel5/ProFSAM-Fire-Detector": {
        "url": "https://huggingface.co/UEmmanuel5/ProFSAM-Fire-Detector/resolve/main/Fire_best.pt",
        "filename": "Fire_best_ProFSAM_yolo11n.pt",
        "description": "Modelo YOLO11n fire-only do projeto ProFSAM; peso Fire_best.pt.",
    },
    "YOLOv8n D-Fire antigo — Hugging Face rabahdev/fire-smoke-yolov8n": {
        "url": "https://huggingface.co/rabahdev/fire-smoke-yolov8n/resolve/main/best.pt",
        "filename": "fire_smoke_yolov8n_dfire_best.pt",
        "description": "Modelo YOLOv8n ajustado no D-Fire; classes: smoke=0, fire=1. Mantido apenas como fallback.",
    },
    "URL manual de arquivo .pt": {
        "url": "",
        "filename": "fire_model_manual.pt",
        "description": "Use esta opção se você tiver outro link direto para um arquivo .pt.",
    },
}


def log_msg(msg, tipo="info"):
    cores = {
        "info": "#1f4e79",
        "ok": "#1f7a1f",
        "erro": "#a61b1b",
        "aviso": "#946200",
    }
    cor = cores.get(tipo, "#333")
    display(HTML(f"<div style='margin:4px 0;color:{cor}'>{msg}</div>"))


def instalar_pacote(nome_pip, nome_import=None):
    """Instala pacote no kernel atual apenas se o import falhar."""
    nome_import = nome_import or nome_pip
    try:
        __import__(nome_import)
        return True
    except Exception:
        log_msg(f"Instalando pacote <code>{nome_pip}</code>...", "aviso")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", nome_pip])
        except Exception as e:
            log_msg(f"Falha ao instalar <code>{nome_pip}</code>: <code>{e}</code>", "erro")
            return False
        try:
            __import__(nome_import)
            return True
        except Exception as e:
            log_msg(f"Pacote instalado, mas o import ainda falhou: <code>{e}</code>", "erro")
            return False


def garantir_ultralytics():
    ok = instalar_pacote("ultralytics", "ultralytics")
    return bool(ok)


def baixar_arquivo(url, destino):
    """Baixa um arquivo com barra simples de progresso textual."""
    destino = Path(destino).resolve()
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.exists() and destino.stat().st_size > 1024 * 1024:
        log_msg(f"Arquivo já existe: <code>{destino}</code>", "ok")
        return destino

    log_msg(f"Baixando modelo de:<br><code>{url}</code>")
    log_msg(f"Destino:<br><code>{destino}</code>")

    def hook(block_num, block_size, total_size):
        if total_size <= 0:
            return
        downloaded = min(block_num * block_size, total_size)
        pct = downloaded * 100.0 / total_size
        if block_num % 25 == 0 or downloaded >= total_size:
            print(f"\rDownload: {pct:5.1f}%", end="")

    try:
        urllib.request.urlretrieve(url, destino, reporthook=hook)
        print()
    except Exception as e:
        if destino.exists():
            try:
                destino.unlink()
            except Exception:
                pass
        log_msg(f"Falha no download: <code>{e}</code>", "erro")
        return None

    if not destino.exists() or destino.stat().st_size < 1024 * 1024:
        log_msg("O arquivo baixado parece pequeno demais para ser um peso YOLO válido.", "erro")
        return None

    log_msg(f"Download concluído: <code>{destino.name}</code> ({destino.stat().st_size/1024/1024:.1f} MB)", "ok")
    return destino


def registrar_modelo_fogo(caminho_modelo, copiar_para_padrao=True):
    """Define o modelo de fogo que será usado pelo restante do notebook."""
    global FIRE_MODEL_READY, DEFAULT_FIRE_MODEL_PATH

    caminho_modelo = Path(caminho_modelo).expanduser().resolve()
    if not caminho_modelo.exists():
        log_msg(f"Modelo não encontrado: <code>{caminho_modelo}</code>", "erro")
        return None
    if caminho_modelo.suffix.lower() != ".pt":
        log_msg("O arquivo selecionado precisa ser um peso PyTorch/Ultralytics com extensão .pt.", "erro")
        return None

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    destino_padrao = Path(DEFAULT_FIRE_MODEL_PATH).resolve()

    if copiar_para_padrao and caminho_modelo != destino_padrao:
        shutil.copy2(caminho_modelo, destino_padrao)
        FIRE_MODEL_READY = destino_padrao
        log_msg(f"Modelo registrado como padrão:<br><code>{destino_padrao}</code>", "ok")
    else:
        FIRE_MODEL_READY = caminho_modelo
        log_msg(f"Modelo registrado:<br><code>{FIRE_MODEL_READY}</code>", "ok")

    return FIRE_MODEL_READY


def validar_modelo_fogo(caminho_modelo=None):
    """Tenta carregar o modelo com Ultralytics e mostra as classes disponíveis."""
    caminho_modelo = Path(caminho_modelo or FIRE_MODEL_READY or DEFAULT_FIRE_MODEL_PATH).expanduser().resolve()

    if not caminho_modelo.exists():
        log_msg(f"Modelo não encontrado: <code>{caminho_modelo}</code>", "erro")
        return False

    if not garantir_ultralytics():
        return False

    try:
        from ultralytics import YOLO
        model = YOLO(str(caminho_modelo))
        names = getattr(model, "names", {})
        log_msg(f"Modelo carregado com sucesso:<br><code>{caminho_modelo}</code>", "ok")
        log_msg(f"Classes do modelo: <code>{names}</code>", "info")
        if isinstance(names, dict):
            nomes_lower = [str(v).lower() for v in names.values()]
        else:
            nomes_lower = [str(v).lower() for v in names]
        if not any(("fire" in n) or ("flame" in n) or ("fogo" in n) or ("chama" in n) for n in nomes_lower):
            log_msg("Atenção: não encontrei uma classe com nome fire/flame/fogo/chama. O notebook tentará usar as detecções mesmo assim se houver uma única classe.", "aviso")
        return True
    except Exception as e:
        log_msg(f"Falha ao carregar o modelo com Ultralytics: <code>{e}</code>", "erro")
        return False


def baixar_modelo_pronto(nome_opcao, url_manual=""):
    """Baixa modelo pronto e registra em DEFAULT_FIRE_MODEL_PATH."""
    info = PRETRAINED_FIRE_MODELS[nome_opcao]
    url = url_manual.strip() if nome_opcao == "URL manual de arquivo .pt" else info["url"]

    if not url:
        log_msg("Informe uma URL direta para um arquivo .pt ou escolha o modelo padrão.", "erro")
        return None
    if not url.lower().split("?")[0].endswith(".pt") and "huggingface.co" not in url.lower():
        log_msg("A URL não parece apontar para um arquivo .pt. Verifique se é link direto para pesos do modelo.", "aviso")

    filename = info["filename"]
    if nome_opcao == "URL manual de arquivo .pt":
        filename = Path(url.split("?")[0]).name or "fire_model_manual.pt"
        if not filename.lower().endswith(".pt"):
            filename = "fire_model_manual.pt"

    destino = MODELS_DIR / filename
    baixado = baixar_arquivo(url, destino)
    if baixado is None:
        return None

    registrado = registrar_modelo_fogo(baixado, copiar_para_padrao=True)
    if registrado is not None:
        validar_modelo_fogo(registrado)
    return registrado


def procurar_modelos_locais():
    modelos = sorted(MODELS_DIR.glob("*.pt"))
    return [str(p) for p in modelos]


# ------------------------------------------------------------
# Interface da célula
# ------------------------------------------------------------
modelo_pronto_w = widgets.Dropdown(
    options=list(PRETRAINED_FIRE_MODELS.keys()),
    value="YOLO11n Fire — Hugging Face UEmmanuel5/ProFSAM-Fire-Detector",
    description="Modelo:",
    layout=widgets.Layout(width="700px"),
)
url_manual_w = widgets.Text(
    value="",
    description="URL .pt:",
    placeholder="Cole aqui somente se escolher URL manual",
    layout=widgets.Layout(width="700px"),
)
modelo_local_w = widgets.Dropdown(
    options=procurar_modelos_locais() or ["<nenhum modelo .pt em models/>"] ,
    description="Local:",
    layout=widgets.Layout(width="700px"),
)
btn_baixar_modelo = widgets.Button(description="Baixar/registrar modelo pronto", button_style="success", icon="download")
btn_atualizar_modelos = widgets.Button(description="Atualizar modelos locais", button_style="", icon="refresh")
btn_usar_local = widgets.Button(description="Usar modelo local selecionado", button_style="primary", icon="check")
btn_validar = widgets.Button(description="Validar modelo padrão", button_style="info", icon="search")
out_modelo = widgets.Output()


def on_baixar_modelo(_):
    with out_modelo:
        clear_output(wait=True)
        baixar_modelo_pronto(modelo_pronto_w.value, url_manual_w.value)


def on_atualizar_modelos(_):
    modelos = procurar_modelos_locais()
    modelo_local_w.options = modelos or ["<nenhum modelo .pt em models/>"]
    with out_modelo:
        clear_output(wait=True)
        if modelos:
            log_msg("Modelos encontrados em <code>models/</code>:", "ok")
            for m in modelos:
                log_msg(f"<code>{m}</code>")
        else:
            log_msg(f"Nenhum arquivo .pt encontrado em:<br><code>{MODELS_DIR}</code>", "aviso")


def on_usar_local(_):
    with out_modelo:
        clear_output(wait=True)
        selecionado = str(modelo_local_w.value)
        if selecionado.startswith("<"):
            log_msg("Nenhum modelo local selecionado.", "erro")
            return
        registrado = registrar_modelo_fogo(selecionado, copiar_para_padrao=True)
        if registrado is not None:
            validar_modelo_fogo(registrado)


def on_validar(_):
    with out_modelo:
        clear_output(wait=True)
        validar_modelo_fogo(DEFAULT_FIRE_MODEL_PATH)


btn_baixar_modelo.on_click(on_baixar_modelo)
btn_atualizar_modelos.on_click(on_atualizar_modelos)
btn_usar_local.on_click(on_usar_local)
btn_validar.on_click(on_validar)

info_modelo = widgets.HTML(f"""
<div style='line-height:1.45; max-width:980px'>
<b>Fluxo desta célula:</b><br>
1. Clique em <b>Baixar/registrar modelo pronto</b> para baixar o modelo padrão de fogo/fumaça.<br>
2. O arquivo será salvo em <code>{MODELS_DIR}</code> e copiado para <code>{DEFAULT_FIRE_MODEL_PATH}</code>.<br>
3. Na célula 3, o campo <b>Modelo fogo</b> já aponta para esse caminho padrão.<br><br>
<b>Observação:</b> a classe <code>smoke</code> pode aparecer no modelo, mas o jato só usa detecções cujo rótulo pareça <code>fire/flame/fogo/chama</code>.
</div>
""")

ui_modelo = widgets.VBox([
    info_modelo,
    widgets.HBox([btn_baixar_modelo, btn_validar]),
    modelo_pronto_w,
    url_manual_w,
    widgets.HBox([btn_atualizar_modelos, btn_usar_local]),
    modelo_local_w,
    out_modelo,
])

display(ui_modelo)

# Se já existir um modelo padrão, registra automaticamente para facilitar reexecuções.
if Path(DEFAULT_FIRE_MODEL_PATH).exists():
    FIRE_MODEL_READY = Path(DEFAULT_FIRE_MODEL_PATH).resolve()
    with out_modelo:
        log_msg(f"Modelo padrão já encontrado:<br><code>{FIRE_MODEL_READY}</code>", "ok")


In [3]:
# ============================================================
# CÉLULA 3 - Configuração-base e setups
# ============================================================
# Nesta versão, os sliders ficam na CÉLULA 5 para que os ajustes reflitam
# em tempo real no vídeo. Esta célula mantém apenas:
# - valores padrão;
# - explicações curtas dos parâmetros;
# - funções de salvar/carregar setups em JSON;
# - pasta padrão para vídeos de entrada.
# ============================================================

import json
from pathlib import Path
from IPython.display import display, HTML

VIDEO_INPUT_DIR = BASE_DIR / "videos_input"
VIDEO_INPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_FIRE_DETECTIONS_INTERNAL = 300  # considerado "ilimitado" para uso prático em webcam/vídeo.
MAX_PERSON_DETECTIONS_INTERNAL = 8
MIN_FIRE_AREA_INTERNAL = 0

DEFAULT_CONFIG = {
    "setup_name": "rgb_monocular_rotas_v8",

    # Fonte de vídeo
    "source_mode": "camera",        # "camera" ou "video"
    "camera_index": 0,
    "video_file": "",
    "video_loop": True,
    "cap_width": 640,
    "cap_height": 480,
    "cap_fps": 30,

    # Modelos
    "fire_model_path": str(DEFAULT_FIRE_MODEL_PATH),
    "person_model_path": DEFAULT_PERSON_MODEL_NAME,

    # Inferência
    "fire_conf": 0.10,
    "person_conf": 0.40,
    "imgsz_infer": 960,
    "detect_every_n_frames": 2,
    "max_fire_detections": MAX_FIRE_DETECTIONS_INTERNAL,
    "max_person_detections": MAX_PERSON_DETECTIONS_INTERNAL,
    "min_fire_area_px": MIN_FIRE_AREA_INTERNAL,

    # Planejamento automático do jato
    "large_region_area_px": 2500,
    "lower_edge_offset_px": 8,
    "route_step_px": 18,
    "route_speed_px_frame": 14,
    "transition_speed_px_frame": 35,
    "centroid_hold_frames": 5,

    # Queda do jato / pressão
    "jet_drop_px": 20,
    "jet_pressure_calib": 22.0,

    # Segurança e apagamento simulado
    "human_safety_radius_px": 90,
    "human_safety_use_bbox_distance": True,
    "erase_radius_px": 28,
    "erased_fraction_threshold": 0.45,
    "simulate_extinguish": True,
    "stop_if_no_fire": True,

    # Arduino
    "arduino_port": "",
    "arduino_baud": 115200,
    "send_to_arduino": True,
    "authorize_real_water": True,

    # Fallback linear se calibração 3 pontos ainda não foi aplicada
    "servo_pan_min": 25,
    "servo_pan_max": 155,
    "servo_tilt_min": 35,
    "servo_tilt_max": 145,
}

PARAM_HELP = {
    "fire_conf": "Confiança mínima da detecção de chama. Menor detecta focos fracos, mas aumenta falsos positivos.",
    "person_conf": "Confiança mínima para detectar humano/pessoa pelo YOLO COCO.",
    "imgsz_infer": "Tamanho usado pelo YOLO na inferência. Maior ajuda foco pequeno, mas reduz FPS.",
    "detect_every_n_frames": "Frequência da inferência. 1 detecta todo frame; 2 detecta frame sim/frame não.",
    "large_region_area_px": "Área, em pixels, acima da qual a chama usa rota pela borda inferior.",
    "lower_edge_offset_px": "Distância vertical abaixo da borda inferior da chama para formar os pontos de impacto.",
    "route_step_px": "Espaçamento entre pontos na borda inferior da chama grande.",
    "route_speed_px_frame": "Velocidade da mira ao percorrer pontos consecutivos da mesma trajetória.",
    "transition_speed_px_frame": "Velocidade da mira ao saltar entre trajetórias ou centróides diferentes.",
    "jet_drop_px": "Compensação da queda do jato: a mira enviada ao servo fica esta quantidade de pixels acima do impacto.",
    "erase_radius_px": "Raio da região apagada assim que o ponto de impacto passa por ela.",
    "human_safety_radius_px": "Raio ao redor do impacto usado para bloquear o jato se houver humano próximo.",
    "erased_fraction_threshold": "Fração da caixa da chama que precisa estar coberta para ser considerada suprimida na simulação.",
}


def carregar_todos_setups():
    if not FIRE_SETUPS_FILE.exists():
        return {}
    try:
        with open(FIRE_SETUPS_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


def salvar_todos_setups(data):
    FIRE_SETUPS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(FIRE_SETUPS_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def listar_nomes_setups():
    return sorted(carregar_todos_setups().keys())


def mesclar_config(cfg):
    """Completa configs antigas com os padrões atuais."""
    return {**DEFAULT_CONFIG, **(cfg or {})}


def salvar_setup_config(nome, cfg, servo_calibration=None):
    nome = (nome or "setup_sem_nome").strip()
    data = carregar_todos_setups()
    payload = {
        "config": mesclar_config(cfg),
        "servo_calibration": servo_calibration or globals().get("SERVO_CALIBRATION", None),
        "saved_at": datetime.now().isoformat(timespec="seconds") if "datetime" in globals() else "",
    }
    data[nome] = payload
    salvar_todos_setups(data)
    return nome


def carregar_setup_config(nome):
    data = carregar_todos_setups()
    item = data.get(nome, {})
    if isinstance(item, dict) and "config" in item:
        return mesclar_config(item.get("config", {})), item.get("servo_calibration")
    # compatibilidade com setups antigos salvos como dicionário direto
    if isinstance(item, dict):
        return mesclar_config(item), None
    return mesclar_config({}), None


def html_parametros_resumo():
    linhas = [
        "<b>Célula 3 carregada.</b>",
        "Os ajustes interativos foram movidos para a <b>Célula 5</b>, junto do vídeo, para alteração em tempo real.",
        f"<b>Setups:</b> <code>{FIRE_SETUPS_FILE}</code>",
        f"<b>Vídeos de entrada:</b> <code>{VIDEO_INPUT_DIR}</code>",
        "Coloque arquivos <code>.mp4</code>, <code>.avi</code>, <code>.mov</code> ou <code>.mkv</code> nessa pasta para usar vídeo em vez de webcam.",
    ]
    return "<br>".join(linhas)


display(HTML(html_parametros_resumo()))

In [4]:
# ============================================================
# CÉLULA 4 - Funções auxiliares: YOLO, rotas, segurança, painéis e Arduino
# ============================================================

import math
import time
import json
import asyncio
import subprocess
import sys
from pathlib import Path

import numpy as np
import cv2
from IPython.display import display, HTML

# ------------------------------------------------------------
# Utilidades visuais
# ------------------------------------------------------------
def bgr_to_jpeg_bytes(img_bgr, quality=85):
    if img_bgr is None or img_bgr.size == 0:
        img_bgr = np.zeros((240, 320, 3), dtype=np.uint8)
    ok, buf = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    if not ok:
        return b""
    return buf.tobytes()


def texto_ascii_cv2(text):
    """OpenCV putText não renderiza acentos; simplifica texto antes de desenhar."""
    mapa = str.maketrans(
        "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ",
        "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC",
    )
    return str(text).translate(mapa)


def put_label(img, text, org, color=(255, 255, 255), bg=(0, 0, 0), scale=0.55, thickness=1):
    x, y = int(org[0]), int(org[1])
    txt = texto_ascii_cv2(text)
    (tw, th), base = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)
    cv2.rectangle(img, (x, y - th - base - 4), (x + tw + 6, y + 4), bg, -1)
    cv2.putText(img, txt, (x + 3, y - 3), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)


def criar_canvas_vazio(w=640, h=360, texto="sem imagem"):
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[:] = (35, 35, 35)
    cv2.putText(img, texto_ascii_cv2(texto), (30, h // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (230, 230, 230), 2, cv2.LINE_AA)
    return img


def clip_box(box, w, h):
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    x1 = max(0, min(w - 1, x1))
    x2 = max(0, min(w - 1, x2))
    y1 = max(0, min(h - 1, y1))
    y2 = max(0, min(h - 1, y2))
    if x2 <= x1:
        x2 = min(w - 1, x1 + 1)
    if y2 <= y1:
        y2 = min(h - 1, y1 + 1)
    return x1, y1, x2, y2


# ------------------------------------------------------------
# YOLO
# ------------------------------------------------------------
def carregar_modelo_yolo(model_path_or_name):
    from ultralytics import YOLO
    return YOLO(str(model_path_or_name))


def carregar_modelos_runtime(cfg):
    fire_model_path = cfg.get("fire_model_path", str(DEFAULT_FIRE_MODEL_PATH))
    person_model_path = cfg.get("person_model_path", DEFAULT_PERSON_MODEL_NAME)

    fire_path = Path(fire_model_path)
    fire_model = carregar_modelo_yolo(str(fire_path) if fire_path.exists() else fire_model_path)
    person_model = carregar_modelo_yolo(person_model_path)
    return fire_model, person_model


def nomes_modelo(model):
    names = getattr(model, "names", {})
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {i: str(v) for i, v in enumerate(names)}


def encontrar_classes_por_palavras(model, palavras):
    nomes = nomes_modelo(model)
    achadas = []
    palavras = [p.lower() for p in palavras]
    for idx, nome in nomes.items():
        n = nome.lower().strip()
        if any(p in n for p in palavras):
            achadas.append(idx)
    return sorted(achadas)


def detectar_yolo(frame_bgr, model, conf=0.35, imgsz=640, classes=None, max_det=300):
    results = model.predict(
        frame_bgr,
        conf=float(conf),
        imgsz=int(imgsz),
        classes=classes,
        max_det=int(max_det),
        verbose=False,
    )
    dets = []
    nomes = nomes_modelo(model)
    if not results:
        return dets
    r = results[0]
    if r.boxes is None:
        return dets

    xyxy = r.boxes.xyxy.detach().cpu().numpy()
    confs = r.boxes.conf.detach().cpu().numpy()
    clss = r.boxes.cls.detach().cpu().numpy().astype(int)
    h, w = frame_bgr.shape[:2]

    for box, cf, cls_id in zip(xyxy, confs, clss):
        x1, y1, x2, y2 = clip_box(box, w, h)
        area = max(0, x2 - x1) * max(0, y2 - y1)
        dets.append({
            "box": (x1, y1, x2, y2),
            "conf": float(cf),
            "cls": int(cls_id),
            "label": nomes.get(int(cls_id), str(cls_id)),
            "center": ((x1 + x2) / 2.0, (y1 + y2) / 2.0),
            "area": float(area),
        })
    return dets


def filtrar_fogos_por_nome_e_area(dets, min_area=0):
    palavras_fogo = ["fire", "flame", "chama", "fogo"]
    filtrados = []
    for d in dets:
        label = d.get("label", "").lower()
        if d.get("area", 0) < min_area:
            continue
        if any(p in label for p in palavras_fogo):
            filtrados.append(d)

    # Se o modelo pronto tiver uma única classe com outro nome, não descarta tudo.
    if not filtrados and len(dets) > 0:
        filtrados = [d for d in dets if d.get("area", 0) >= min_area]
    return filtrados


# ------------------------------------------------------------
# Planejamento: borda inferior para regiões grandes + centróides para remanescentes
# ------------------------------------------------------------
def criar_mapa_hits(frame_shape):
    h, w = frame_shape[:2]
    return np.zeros((h, w), dtype=np.uint8)


def registrar_hit_apagamento(hit_map, impact, radius):
    """Apaga imediatamente a região percorrida pelo ponto de impacto."""
    if hit_map is None or impact is None:
        return hit_map
    x, y = int(round(impact[0])), int(round(impact[1]))
    cv2.circle(hit_map, (x, y), int(radius), 1, -1)
    return hit_map


def overlay_regioes_apagadas(img, hit_map):
    if hit_map is None:
        return img
    mask = hit_map > 0
    if not np.any(mask):
        return img
    out = img.copy()
    overlay = out.copy()
    overlay[mask] = (110, 110, 110)
    cv2.addWeighted(overlay, 0.45, out, 0.55, 0, out)
    return out


def fire_erased_fraction(fire_det, hit_map):
    if hit_map is None:
        return 0.0
    h, w = hit_map.shape[:2]
    x1, y1, x2, y2 = clip_box(fire_det["box"], w, h)
    roi = hit_map[y1:y2 + 1, x1:x2 + 1]
    if roi.size == 0:
        return 0.0
    return float(np.mean(roi > 0))


def filtrar_fogos_nao_apagados(fires, hit_map, threshold=0.45):
    ativos = []
    apagados = []
    for f in fires:
        frac = fire_erased_fraction(f, hit_map)
        item = dict(f)
        item["erased_fraction"] = frac
        if frac >= float(threshold):
            apagados.append(item)
        else:
            ativos.append(item)
    return ativos, apagados


def pontos_borda_inferior(fire_det, cfg, frame_shape):
    h, w = frame_shape[:2]
    x1, y1, x2, y2 = fire_det["box"]
    step = max(2, int(cfg.get("route_step_px", 18)))
    offset = int(cfg.get("lower_edge_offset_px", 8))
    yy = int(np.clip(y2 + offset, 0, h - 1))
    pts = []
    if x2 <= x1:
        return [(int((x1 + x2) / 2), yy)]
    for xx in range(int(x1), int(x2) + 1, step):
        pts.append((int(np.clip(xx, 0, w - 1)), yy))
    if pts[-1][0] != int(x2):
        pts.append((int(np.clip(x2, 0, w - 1)), yy))
    return pts


def ponto_centroide(fire_det, frame_shape):
    h, w = frame_shape[:2]
    cx, cy = fire_det.get("center", (0, 0))
    return (int(np.clip(round(cx), 0, w - 1)), int(np.clip(round(cy), 0, h - 1)))


def planejar_rota_combate(fires, hit_map, cfg, frame_shape):
    """
    Estratégia automática:
    1. Enquanto houver chamas grandes ativas, percorre a borda inferior delas.
    2. Quando restarem apenas chamas pequenas, conecta os centróides.
    """
    large_thr = int(cfg.get("large_region_area_px", 2500))
    erase_thr = float(cfg.get("erased_fraction_threshold", 0.45))
    fires_active, fires_erased = filtrar_fogos_nao_apagados(fires, hit_map, erase_thr)

    large = [f for f in fires_active if f.get("area", 0) >= large_thr]
    small = [f for f in fires_active if f.get("area", 0) < large_thr]

    route = []
    meta = []

    if large:
        large = sorted(large, key=lambda f: (f.get("area", 0), f.get("conf", 0)), reverse=True)
        for reg_idx, f in enumerate(large):
            pts = pontos_borda_inferior(f, cfg, frame_shape)
            # alterna sentido para reduzir deslocamento quando houver várias chamas grandes
            if reg_idx % 2 == 1:
                pts = list(reversed(pts))
            for p in pts:
                route.append(p)
                meta.append({"mode": "borda_inferior", "fire": f})
        mode = "borda_inferior"
    else:
        small = sorted(small, key=lambda f: (f["center"][0], f["center"][1]))
        hold = max(1, int(cfg.get("centroid_hold_frames", 5)))
        for f in small:
            c = ponto_centroide(f, frame_shape)
            for _ in range(hold):
                route.append(c)
                meta.append({"mode": "centroide", "fire": f})
        mode = "centroides"

    return {
        "route": route,
        "meta": meta,
        "mode": mode if route else "sem_rota",
        "active_fires": fires_active,
        "erased_fires": fires_erased,
        "large_count": len(large),
        "small_count": len(small),
    }


def assinatura_rota(route):
    if not route:
        return ()
    # Assinatura leve para detectar mudança relevante sem ficar sensível a pequenas variações de 1 px.
    return tuple((int(round(x / 4) * 4), int(round(y / 4) * 4)) for x, y in route[:80]) + ((len(route), len(route)),)


def move_point_towards(current, target, max_step):
    if current is None:
        return (float(target[0]), float(target[1])), True
    cx, cy = current
    tx, ty = float(target[0]), float(target[1])
    dx, dy = tx - cx, ty - cy
    d = float(math.hypot(dx, dy))
    if d <= max_step or d < 1e-6:
        return (tx, ty), True
    s = max_step / d
    return (cx + dx * s, cy + dy * s), False


def atualizar_mira_por_rota(state, plan, cfg):
    route = plan.get("route", [])
    meta = plan.get("meta", [])
    if not route:
        state["route_signature"] = None
        state["route_index"] = 0
        state["impact"] = None
        state["aim"] = None
        state["mode"] = "sem_rota"
        return state

    sig = assinatura_rota(route)
    route_changed = sig != state.get("route_signature")

    if route_changed:
        state["route_signature"] = sig
        # Ao mudar de rota, escolhe o ponto mais próximo da mira atual para evitar saltos desnecessários.
        cur = state.get("impact") or route[0]
        idx = int(np.argmin([math.hypot(p[0] - cur[0], p[1] - cur[1]) for p in route])) if route else 0
        state["route_index"] = max(0, min(idx, len(route) - 1))
        state["transitioning"] = True

    idx = max(0, min(int(state.get("route_index", 0)), len(route) - 1))
    target = route[idx]
    mode = meta[idx].get("mode", "rota") if idx < len(meta) else "rota"

    if state.get("transitioning", False) or mode == "centroide":
        speed = float(cfg.get("transition_speed_px_frame", 35))
    else:
        speed = float(cfg.get("route_speed_px_frame", 14))

    new_impact, reached = move_point_towards(state.get("impact"), target, speed)
    state["impact"] = new_impact
    state["mode"] = mode

    if reached:
        state["transitioning"] = False
        if idx < len(route) - 1:
            state["route_index"] = idx + 1
        else:
            # Mantém varredura cíclica enquanto a chama persistir.
            state["route_index"] = 0
            state["transitioning"] = True

    aim, drop = calcular_mira_compensada(new_impact, cfg, frame_shape=state.get("frame_shape"))
    state["aim"] = aim
    state["drop_px"] = drop
    return state


def calcular_mira_compensada(impact, cfg, frame_shape=None):
    """Se o impacto real fica abaixo da mira pela gravidade, mira acima do impacto."""
    if impact is None:
        return None, 0.0
    x, y = float(impact[0]), float(impact[1])
    drop = float(cfg.get("jet_drop_px", 20))
    aim_y = y - drop
    if frame_shape is not None:
        h, w = frame_shape[:2]
        x = float(np.clip(x, 0, w - 1))
        aim_y = float(np.clip(aim_y, 0, h - 1))
    return (x, aim_y), drop


# ------------------------------------------------------------
# Segurança: humano perto do impacto real
# ------------------------------------------------------------
def distancia_ponto_bbox(p, box):
    px, py = p
    x1, y1, x2, y2 = box
    dx = max(x1 - px, 0, px - x2)
    dy = max(y1 - py, 0, py - y2)
    return math.hypot(dx, dy)


def humano_perto_do_impacto(persons, impact, raio_px, usar_bbox=True):
    if impact is None:
        return False, None, None
    melhor = None
    melhor_dist = float("inf")
    for p in persons:
        if usar_bbox:
            dist = distancia_ponto_bbox(impact, p["box"])
        else:
            cx, cy = p["center"]
            dist = math.hypot(cx - impact[0], cy - impact[1])
        if dist < melhor_dist:
            melhor = p
            melhor_dist = dist
    bloqueado = melhor is not None and melhor_dist <= float(raio_px)
    return bloqueado, melhor, melhor_dist


# ------------------------------------------------------------
# Painéis
# ------------------------------------------------------------
def desenhar_painel_original(frame, fires, persons, impact, safety_radius, blocked, status_text):
    out = frame.copy()
    for f in fires:
        x1, y1, x2, y2 = f["box"]
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 80, 255), 2)
        put_label(out, f"{f['label']} {f['conf']:.2f} A={int(f['area'])}", (x1, max(18, y1)), bg=(0, 45, 120))

    for p in persons:
        x1, y1, x2, y2 = p["box"]
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 210, 0), 2)
        put_label(out, f"humano {p['conf']:.2f}", (x1, max(18, y1)), bg=(0, 100, 0))

    if impact is not None:
        ix, iy = int(round(impact[0])), int(round(impact[1]))
        cv2.circle(out, (ix, iy), int(safety_radius), (0, 255, 255), 2)
        cv2.drawMarker(out, (ix, iy), (0, 255, 255), markerType=cv2.MARKER_CROSS, markerSize=20, thickness=2)
        put_label(out, "impacto", (ix + 8, max(18, iy - 8)), bg=(80, 40, 0))

    cor_bg = (0, 0, 150) if blocked else (0, 100, 0)
    put_label(out, status_text, (10, 28), bg=cor_bg, scale=0.65, thickness=2)
    return out


def desenhar_painel_combate(frame, plan, aim_state, persons, hit_map, cfg, blocked=False):
    out = frame.copy()
    if cfg.get("simulate_extinguish", True):
        out = overlay_regioes_apagadas(out, hit_map)

    route = plan.get("route", [])
    active = plan.get("active_fires", [])
    erased = plan.get("erased_fires", [])
    impact = aim_state.get("impact")
    aim = aim_state.get("aim")
    idx = int(aim_state.get("route_index", 0))

    for f in active:
        x1, y1, x2, y2 = f["box"]
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 80, 255), 2)
    for f in erased:
        x1, y1, x2, y2 = f["box"]
        cv2.rectangle(out, (x1, y1), (x2, y2), (130, 130, 130), 1)
        put_label(out, "sim apagado", (x1, max(18, y1)), bg=(80, 80, 80), scale=0.45)

    if route:
        pts = np.array([(int(x), int(y)) for x, y in route], dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(out, [pts], False, (255, 0, 255), 2)
        for k, p in enumerate(route):
            color = (255, 0, 255) if k >= idx else (90, 90, 90)
            cv2.circle(out, (int(p[0]), int(p[1])), 3, color, -1)

    if impact is not None:
        ix, iy = int(round(impact[0])), int(round(impact[1]))
        cv2.circle(out, (ix, iy), int(cfg.get("erase_radius_px", 28)), (0, 255, 255), 2)
        cv2.drawMarker(out, (ix, iy), (0, 255, 255), markerType=cv2.MARKER_TILTED_CROSS, markerSize=22, thickness=2)
        put_label(out, "impacto real", (ix + 8, max(18, iy - 8)), bg=(80, 45, 0), scale=0.50)

    if aim is not None:
        ax, ay = int(round(aim[0])), int(round(aim[1]))
        cv2.drawMarker(out, (ax, ay), (0, 0, 255), markerType=cv2.MARKER_CROSS, markerSize=26, thickness=2)
        put_label(out, "mira servo", (ax + 8, max(18, ay - 8)), bg=(70, 0, 0), scale=0.50)
        if impact is not None:
            cv2.line(out, (ax, ay), (int(round(impact[0])), int(round(impact[1]))), (0, 180, 255), 1)

    mode = plan.get("mode", "sem_rota")
    status = f"rota: {mode} | ativas={len(active)} apagadas={len(erased)}"
    if blocked:
        status += " | BLOQUEADO humano"
    put_label(out, status, (10, 28), bg=(0, 0, 150) if blocked else (80, 40, 110), scale=0.60, thickness=2)
    put_label(out, f"queda={float(cfg.get('jet_drop_px', 0)):.0f}px", (10, 58), bg=(60, 60, 60), scale=0.50)
    return out


# ------------------------------------------------------------
# Arduino e calibração dos servos
# ------------------------------------------------------------
def listar_portas_serial():
    try:
        import serial
        from serial.tools import list_ports
        return [p.device for p in list_ports.comports()]
    except Exception:
        return []


def abrir_serial_arduino(porta, baud=115200, timeout=0.1):
    import serial
    porta = str(porta).strip()
    if not porta:
        raise ValueError("Porta serial vazia. Ex.: /dev/ttyACM0, /dev/ttyUSB0 ou COM3.")
    ser = serial.Serial(porta, int(baud), timeout=timeout)
    time.sleep(2.0)
    try:
        ser.reset_input_buffer()
        ser.reset_output_buffer()
    except Exception:
        pass
    return ser


def mapear_servo(v, in_min, in_max, out_min, out_max):
    if in_max == in_min:
        return float((out_min + out_max) / 2)
    t = (float(v) - float(in_min)) / float(in_max - in_min)
    t = max(0.0, min(1.0, t))
    return float(out_min) + t * float(out_max - out_min)


if "SERVO_CALIBRATION" not in globals():
    SERVO_CALIBRATION = {
        "enabled": False,
        "frame_size": [640, 480],
        "image_points": {},
        "servo_points": {},
        "coef_pan": None,
        "coef_tilt": None,
    }


def calcular_mapeamento_afim_3pontos(image_points, servo_points):
    keys = ["tl", "tr", "bl"]
    A, bx, by = [], [], []
    for k in keys:
        x, y = image_points[k]
        pan, tilt = servo_points[k]
        A.append([float(x), float(y), 1.0])
        bx.append(float(pan))
        by.append(float(tilt))
    A = np.asarray(A, dtype=float)
    bx = np.asarray(bx, dtype=float)
    by = np.asarray(by, dtype=float)
    coef_pan = np.linalg.solve(A, bx).tolist()
    coef_tilt = np.linalg.solve(A, by).tolist()
    return coef_pan, coef_tilt


def aplicar_mapeamento_servo_calibrado(x, y, frame_shape, cfg):
    h, w = frame_shape[:2]
    calib = globals().get("SERVO_CALIBRATION", {})
    if calib.get("enabled") and calib.get("coef_pan") is not None and calib.get("coef_tilt") is not None:
        calib_w, calib_h = calib.get("frame_size", [w, h])
        calib_w = float(calib_w or w)
        calib_h = float(calib_h or h)
        x_cal = float(x) * calib_w / max(1.0, float(w))
        y_cal = float(y) * calib_h / max(1.0, float(h))
        a, b, c = [float(v) for v in calib["coef_pan"]]
        d, e, f = [float(v) for v in calib["coef_tilt"]]
        pan = a * x_cal + b * y_cal + c
        tilt = d * x_cal + e * y_cal + f
    else:
        pan = mapear_servo(x, 0, w - 1, cfg.get("servo_pan_min", 25), cfg.get("servo_pan_max", 155))
        tilt = mapear_servo(y, h - 1, 0, cfg.get("servo_tilt_min", 35), cfg.get("servo_tilt_max", 145))
    pan = float(max(0, min(180, pan)))
    tilt = float(max(0, min(180, tilt)))
    return pan, tilt


def ponto_para_servos(point, frame_shape, cfg):
    x, y = point
    return aplicar_mapeamento_servo_calibrado(x, y, frame_shape, cfg)


def enviar_comando_arduino(ser, pan=None, tilt=None, water_on=False, pressure=22.0):
    if ser is None:
        return ""
    if pan is None or tilt is None:
        linha = f"W{1 if water_on else 0}\n"
    else:
        linha = f"X{float(pan):.1f},Y{float(tilt):.1f},P{float(pressure):.1f},W{1 if water_on else 0}\n"
    ser.write(linha.encode("ascii", errors="ignore"))
    try:
        ser.flush()
    except Exception:
        pass
    return linha.strip()


ARDUINO_SKETCH = r"""
#include <Servo.h>

/*
  BLAZE - Controle de mira com dois servos MG90S
  Protocolo serial compatível com o notebook de referência:
    X90.0,Y85.5,P22.0,W0
*/

Servo servoX;
Servo servoY;

const int SERVO_X_PIN = 9;
const int SERVO_Y_PIN = 10;
const int WATER_PIN = 8;
const float SERVO_MIN = 0.0;
const float SERVO_MAX = 180.0;
const char* BLAZE_ID = "UNO_MG90S_RGB_MONO_V8";

float currentX = 90.0;
float currentY = 90.0;
float currentP = 22.0;
bool waterOn = false;
String buffer = "";
unsigned long lastStatusMillis = 0;
const unsigned long STATUS_INTERVAL_MS = 2000;
const unsigned long WATER_TIMEOUT_MS = 1000;
unsigned long lastCommandMillis = 0;

float clampFloat(float value, float minValue, float maxValue) {
  if (value < minValue) return minValue;
  if (value > maxValue) return maxValue;
  return value;
}

void applyServoAngles(float x, float y) {
  currentX = clampFloat(x, SERVO_MIN, SERVO_MAX);
  currentY = clampFloat(y, SERVO_MIN, SERVO_MAX);
  servoX.write((int)round(currentX));
  servoY.write((int)round(currentY));
}

void applyWater(bool state) {
  waterOn = state;
  digitalWrite(WATER_PIN, waterOn ? HIGH : LOW);
}

void printStatusLine() {
  Serial.print("X="); Serial.print(currentX, 1);
  Serial.print(" ; Y="); Serial.print(currentY, 1);
  Serial.print(" ; P="); Serial.print(currentP, 1);
  Serial.print(" ; W="); Serial.print(waterOn ? 1 : 0);
  Serial.print(" ; ID="); Serial.println(BLAZE_ID);
}

void printOk() {
  Serial.print("OK ");
  printStatusLine();
}

String valueUntilCommaOrEnd(String cmd, int startIndex) {
  int comma = cmd.indexOf(',', startIndex);
  if (comma < 0) return cmd.substring(startIndex);
  return cmd.substring(startIndex, comma);
}

bool parseCommand(String cmd) {
  cmd.trim();
  if (cmd.length() == 0) return true;

  if (cmd == "PING") { Serial.print("BLAZE_OK "); Serial.println(BLAZE_ID); return true; }
  if (cmd == "ID?") { Serial.print("BLAZE_ID "); Serial.println(BLAZE_ID); return true; }
  if (cmd == "STATUS?") { printStatusLine(); return true; }
  if (cmd == "ZERO") { applyServoAngles(0.0, 0.0); applyWater(false); printOk(); return true; }
  if (cmd == "CENTER") { applyServoAngles(90.0, 90.0); applyWater(false); printOk(); return true; }

  int wIndex = cmd.indexOf('W');
  if (wIndex >= 0) {
    String wStr = valueUntilCommaOrEnd(cmd, wIndex + 1);
    applyWater(wStr.toInt() == 1);
  }

  int xIndex = cmd.indexOf('X');
  int yIndex = cmd.indexOf('Y');
  int pIndex = cmd.indexOf('P');

  if (xIndex >= 0 && yIndex >= 0) {
    String xStr = valueUntilCommaOrEnd(cmd, xIndex + 1);
    String yStr = valueUntilCommaOrEnd(cmd, yIndex + 1);
    applyServoAngles(xStr.toFloat(), yStr.toFloat());
  }

  if (pIndex >= 0) {
    String pStr = valueUntilCommaOrEnd(cmd, pIndex + 1);
    currentP = pStr.toFloat();
  }

  if (xIndex >= 0 || yIndex >= 0 || pIndex >= 0 || wIndex >= 0) {
    lastCommandMillis = millis();
    printOk();
    return true;
  }
  return false;
}

void setup() {
  Serial.begin(115200);
  servoX.attach(SERVO_X_PIN);
  servoY.attach(SERVO_Y_PIN);
  pinMode(WATER_PIN, OUTPUT);
  applyWater(false);
  applyServoAngles(90.0, 90.0);
  delay(300);
  Serial.println("BLAZE Arduino pronto.");
  Serial.print("ID="); Serial.println(BLAZE_ID);
  Serial.println("Baud=115200");
  Serial.println("Comandos: PING, ID?, STATUS?, ZERO, CENTER, X90.0,Y85.5,P22.0,W0");
  printStatusLine();
  lastStatusMillis = millis();
  lastCommandMillis = millis();
}

void loop() {
  while (Serial.available() > 0) {
    char c = (char)Serial.read();
    if (c == '\n' || c == '\r') {
      if (buffer.length() > 0) {
        bool ok = parseCommand(buffer);
        if (!ok) { Serial.print("ERRO comando: "); Serial.println(buffer); }
        buffer = "";
      }
    } else {
      buffer += c;
      if (buffer.length() > 100) { buffer = ""; Serial.println("ERRO buffer serial muito longo."); }
    }
  }

  if (waterOn && millis() - lastCommandMillis > WATER_TIMEOUT_MS) {
    applyWater(false);
  }

  if (millis() - lastStatusMillis >= STATUS_INTERVAL_MS) {
    printStatusLine();
    lastStatusMillis = millis();
  }
}
"""


def salvar_sketch_arduino():
    pasta = ARDUINO_DIR / "blaze_fire_monitor_rgb_monocular"
    pasta.mkdir(parents=True, exist_ok=True)
    ino = pasta / "blaze_fire_monitor_rgb_monocular.ino"
    ino.write_text(ARDUINO_SKETCH, encoding="utf-8")
    display(HTML(f"Sketch Arduino salvo em:<br><code>{ino}</code>"))
    return ino


salvar_sketch_arduino()
print("Funções auxiliares v8 carregadas.")
print("Portas seriais disponíveis:", listar_portas_serial())

Funções auxiliares v8 carregadas.
Portas seriais disponíveis: ['/dev/ttyACM0']


In [15]:
# ============================================================
# CÉLULA 5 - Execução final unificada: parâmetros em tempo real + vídeo/câmera + Arduino
# ============================================================
# Painéis:
# 1) Original + detecção de chama/humano.
# 2) Imagem completa + estratégia de combate + mira + impacto + regiões apagadas.
# ============================================================

import asyncio
import time
import os
import sys
import glob
import json
import math
import threading
import shutil
import platform
import subprocess
from pathlib import Path

import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ------------------------------------------------------------
# Estado global
# ------------------------------------------------------------
MONITOR_RUNNING = False
MONITOR_TASK = None
SERIAL_HANDLE = None
HIT_MAP = None
LAST_FIRE_DETS = []
LAST_PERSON_DETS = []
LAST_DETECT_FRAME = -999
FRAME_COUNTER = 0
CURRENT_FRAME_SHAPE = (int(DEFAULT_CONFIG["cap_height"]), int(DEFAULT_CONFIG["cap_width"]), 3)
ACTIVE_CALIB_POINT = None
LAST_SENT_COMMAND = ""
LAST_SERIAL_ECHO = ""
AIM_STATE = {"impact": None, "aim": None, "route_index": 0, "route_signature": None, "transitioning": True, "mode": "sem_rota"}

# Planejador congelado: enquanto uma rota esta ativa, a imagem 2 fica congelada
# e o sistema conclui essa rota antes de procurar outra chama.
ROUTE_EXECUTOR = {
    "active": False,
    "plan": None,
    "freeze_frame": None,
    "route_index": 0,
    "impact": None,
    "aim": None,
    "transitioning": True,
    "mode": "sem_rota",
    "done": False,
    "paused_by_human": False,
}
LAST_PLAN = {"route": [], "meta": [], "mode": "sem_rota", "active_fires": [], "erased_fires": []}
LAST_BLOCKED_BY_HUMAN = False
LAST_STATUS_TEXT = "sem chama detectada"
FIRE_ACQUIRE_STATE = {"active": False, "start": 0.0, "dets": [], "last_frame": None, "last_base": None}
FIRE_ACQUIRE_SECONDS = 0.0  # v11/v12: sem espera; congela no primeiro frame com fogo

# Memória curta de humanos: mantém a última posição por 3 s para segurança.
# Isso evita que o jato seja liberado só porque o detector de humano falhou em um ciclo.
HUMAN_MEMORY_SECONDS = 3.0
HUMAN_MEMORY = {"timestamp": 0.0, "persons": []}
LAST_SAFETY_PERSON_DETS = []

# Controle de avanço manual do vídeo por slider.
VIDEO_SEEK_REQUEST = {"frame": None}
VIDEO_SEEK_UPDATING = False

CALIB_POINTS_NORM = {
    "tl": (0.20, 0.20),
    "tr": (0.80, 0.20),
    "bl": (0.20, 0.80),
}
CALIB_LABELS = {
    "tl": "superior esquerdo",
    "tr": "superior direito",
    "bl": "inferior esquerdo",
}
SERVO_CALIBRATION_FILE = USER_SETUPS_DIR / "servo_calibration_3pontos_rgb_monocular.json"

# ------------------------------------------------------------
# Estilo e helpers de UI
# ------------------------------------------------------------
FULL = widgets.Layout(width="100%")
WIDE = widgets.Layout(width="520px")
MED = widgets.Layout(width="360px")
SMALL = widgets.Layout(width="250px")
STYLE = {"description_width": "150px"}

PARAM_HELP = dict(PARAM_HELP)
PARAM_HELP.update({
    "detect_every_n_frames": "Intervalo do ciclo de controle. A imagem 1 usa sempre o frame mais recente; YOLO, seguranca, movimento da mira e serial rodam somente a cada X frames.",
    "route_speed_px_frame": "Velocidade da mira ao percorrer a trajetória. Nesta versão, o valor inicial foi dobrado para acelerar o jato.",
    "transition_speed_px_frame": "Velocidade da mira ao trocar de região ou iniciar uma nova trajetória. Nesta versão, o valor inicial foi dobrado.",
    "jet_drop_px": "Compensacao de queda do jato. Se a agua cai abaixo da mira, aumente este valor para mirar mais acima do impacto desejado.",
    "erase_radius_px": "Raio de atuação do jato na simulação: a região só fica preta quando a mira/impacto passa por ela, não instantaneamente.",
    "erased_fraction_threshold": "Fração da regiao de chama ja pintada de preto para considerar aquele foco apagado e evitar nova rota sobre a mesma area.",
    "human_memory_seconds": "Tempo fixo de memoria do humano: 3 s. Se o humano sumir momentaneamente da deteccao, a ultima bbox ainda bloqueia o jato.",
})

panel1 = widgets.Image(format="jpeg", width=760, height=428)
panel2 = widgets.Image(format="jpeg", width=760, height=428)

status_w = widgets.HTML("<b>Status:</b> parado")
last_command_w = widgets.HTML("<b>Serial:</b> nenhum comando enviado")
calib_status_w = widgets.HTML("<b>Calibração:</b> não aplicada")
out_runtime = widgets.Output(layout=FULL)

btn_start = widgets.Button(description="Iniciar", button_style="success", icon="play", layout=widgets.Layout(width="120px"))
btn_stop = widgets.Button(description="Parar", button_style="danger", icon="stop", layout=widgets.Layout(width="110px"))
btn_reset_erase = widgets.Button(description="Reset apagamento", button_style="warning", icon="refresh", layout=widgets.Layout(width="170px"))

# Setup
setup_name_w = widgets.Text(value=DEFAULT_CONFIG["setup_name"], description="Nome setup", style=STYLE, layout=WIDE)
setup_dropdown_w = widgets.Dropdown(options=["<nenhum>"] + listar_nomes_setups(), description="Carregar", style=STYLE, layout=WIDE)
btn_save_setup = widgets.Button(description="Salvar setup", button_style="success", icon="save", layout=widgets.Layout(width="150px"))
btn_load_setup = widgets.Button(description="Carregar setup", button_style="info", icon="folder-open", layout=widgets.Layout(width="160px"))
btn_refresh_setups = widgets.Button(description="Atualizar lista", icon="refresh", layout=widgets.Layout(width="150px"))

# Fonte de vídeo
source_mode_w = widgets.Dropdown(
    options=[("Câmera em tempo real", "camera"), ("Vídeo da pasta", "video")],
    value=DEFAULT_CONFIG["source_mode"], description="Fonte", style=STYLE, layout=WIDE,
)
btn_list_cameras = widgets.Button(description="Listar câmeras", button_style="info", icon="camera", layout=widgets.Layout(width="160px"))
camera_selector_w = widgets.Dropdown(options=[("Camera 0", "0")], value="0", description="Câmera", style=STYLE, layout=WIDE)
btn_refresh_videos = widgets.Button(description="Listar vídeos", button_style="info", icon="film", layout=widgets.Layout(width="150px"))
video_selector_w = widgets.Dropdown(options=[("Nenhum vídeo encontrado", "")], value="", description="Vídeo", style=STYLE, layout=widgets.Layout(width="720px"))
video_loop_w = widgets.Checkbox(value=True, description="Repetir vídeo ao final", indent=False, layout=widgets.Layout(width="220px"))
video_seek_w = widgets.IntSlider(
    value=0, min=0, max=1, step=1,
    description="Avançar vídeo",
    continuous_update=False,
    disabled=True,
    style=STYLE,
    layout=widgets.Layout(width="100%"),
)
cap_width_w = widgets.IntText(value=DEFAULT_CONFIG["cap_width"], description="Largura", style=STYLE, layout=SMALL)
cap_height_w = widgets.IntText(value=DEFAULT_CONFIG["cap_height"], description="Altura", style=STYLE, layout=SMALL)
cap_fps_w = widgets.IntText(value=DEFAULT_CONFIG["cap_fps"], description="FPS", style=STYLE, layout=SMALL)

# Modelos
fire_model_path_w = widgets.Text(value=DEFAULT_CONFIG["fire_model_path"], description="Modelo chama", style=STYLE, layout=widgets.Layout(width="860px"))
person_model_path_w = widgets.Text(value=DEFAULT_CONFIG["person_model_path"], description="Modelo humano", style=STYLE, layout=widgets.Layout(width="500px"))

# Parâmetros em tempo real
fire_conf_w = widgets.FloatSlider(value=DEFAULT_CONFIG["fire_conf"], min=0.01, max=0.95, step=0.01, description="Conf. chama", readout_format=".2f", continuous_update=True, style=STYLE, layout=MED)
person_conf_w = widgets.FloatSlider(value=DEFAULT_CONFIG["person_conf"], min=0.05, max=0.95, step=0.01, description="Conf. humano", readout_format=".2f", continuous_update=True, style=STYLE, layout=MED)
imgsz_infer_w = widgets.Dropdown(options=[416, 512, 640, 768, 960, 1024, 1280], value=DEFAULT_CONFIG["imgsz_infer"], description="imgsz", style=STYLE, layout=SMALL)
detect_every_w = widgets.IntSlider(value=max(int(DEFAULT_CONFIG["detect_every_n_frames"]), 6), min=1, max=120, step=1, description="Ciclo controle", continuous_update=True, style=STYLE, layout=MED)

large_area_w = widgets.IntSlider(value=DEFAULT_CONFIG["large_region_area_px"], min=50, max=30000, step=50, description="Área chama grande", continuous_update=True, style=STYLE, layout=MED)
lower_offset_w = widgets.IntSlider(value=DEFAULT_CONFIG["lower_edge_offset_px"], min=-80, max=160, step=1, description="Offset borda inf.", continuous_update=True, style=STYLE, layout=MED)
route_step_w = widgets.IntSlider(value=DEFAULT_CONFIG["route_step_px"], min=3, max=80, step=1, description="Passo linha", continuous_update=True, style=STYLE, layout=MED)
route_speed_w = widgets.IntSlider(value=min(240, max(1, int(DEFAULT_CONFIG["route_speed_px_frame"]) * 2)), min=1, max=240, step=1, description="Vel. trajetória", continuous_update=True, style=STYLE, layout=MED)
transition_speed_w = widgets.IntSlider(value=min(320, max(1, int(DEFAULT_CONFIG["transition_speed_px_frame"]) * 2)), min=1, max=320, step=1, description="Vel. transição", continuous_update=True, style=STYLE, layout=MED)
jet_drop_w = widgets.IntSlider(value=DEFAULT_CONFIG["jet_drop_px"], min=0, max=240, step=1, description="Queda jato px", continuous_update=True, style=STYLE, layout=MED)
pressure_w = widgets.FloatSlider(value=DEFAULT_CONFIG["jet_pressure_calib"], min=1.0, max=80.0, step=0.5, description="Pressão P", readout_format=".1f", continuous_update=True, style=STYLE, layout=MED)
erase_radius_w = widgets.IntSlider(value=DEFAULT_CONFIG["erase_radius_px"], min=2, max=160, step=1, description="Raio apagamento", continuous_update=True, style=STYLE, layout=MED)
human_radius_w = widgets.IntSlider(value=DEFAULT_CONFIG["human_safety_radius_px"], min=10, max=400, step=5, description="Raio segurança", continuous_update=True, style=STYLE, layout=MED)
erased_fraction_w = widgets.FloatSlider(value=DEFAULT_CONFIG["erased_fraction_threshold"], min=0.05, max=0.95, step=0.05, description="Fração apagada", readout_format=".2f", continuous_update=True, style=STYLE, layout=MED)

# Arduino
arduino_port_w = widgets.Text(value=DEFAULT_CONFIG["arduino_port"], description="Porta Arduino", placeholder="/dev/ttyACM0, /dev/ttyUSB0 ou COM3", style=STYLE, layout=WIDE)
arduino_baud_w = widgets.IntText(value=DEFAULT_CONFIG["arduino_baud"], description="Baud", style=STYLE, layout=SMALL)
btn_open_sketch = widgets.Button(description="Abrir sketch", button_style="primary", icon="external-link", layout=widgets.Layout(width="140px"))
btn_list_ports = widgets.Button(description="Listar portas", button_style="info", icon="usb", layout=widgets.Layout(width="140px"))
btn_connect_serial = widgets.Button(description="Conectar", button_style="info", icon="plug", layout=widgets.Layout(width="130px"))
btn_disconnect_serial = widgets.Button(description="Desconectar", icon="times", layout=widgets.Layout(width="150px"))
btn_center_servos = widgets.Button(description="Centralizar", icon="crosshairs", layout=widgets.Layout(width="130px"))
btn_ping_arduino = widgets.Button(description="PING/STATUS", icon="exchange", layout=widgets.Layout(width="150px"))
port_selector_w = widgets.Dropdown(options=[("Nenhuma porta detectada", "")], value="", description="Porta", style=STYLE, layout=WIDE)

# Calibração 3 pontos
calib_widgets = {}
for key in ["tl", "tr", "bl"]:
    calib_widgets[key] = {
        "pan": widgets.FloatSlider(value=90.0, min=0, max=180, step=0.5, description="Servo X", readout_format=".1f", continuous_update=True, style={"description_width": "70px"}, layout=widgets.Layout(width="300px")),
        "tilt": widgets.FloatSlider(value=90.0, min=0, max=180, step=0.5, description="Servo Y", readout_format=".1f", continuous_update=True, style={"description_width": "70px"}, layout=widgets.Layout(width="300px")),
        "go": widgets.Button(description="Mirar ponto", icon="crosshairs", layout=widgets.Layout(width="125px")),
    }
btn_apply_calib = widgets.Button(description="Aplicar calibração 3 pontos", button_style="success", icon="check", layout=widgets.Layout(width="230px"))
btn_disable_calib = widgets.Button(description="Desativar calibração", button_style="warning", icon="ban", layout=widgets.Layout(width="190px"))
btn_save_calib = widgets.Button(description="Salvar calibração", icon="save", layout=widgets.Layout(width="170px"))
btn_load_calib = widgets.Button(description="Carregar calibração", icon="folder-open", layout=widgets.Layout(width="190px"))


def param_card(widget, key):
    help_txt = PARAM_HELP.get(key, "")
    return widgets.VBox([
        widget,
        widgets.HTML(f"<div style='font-size:12px;color:#555;line-height:1.15;margin-left:6px'>{help_txt}</div>")
    ], layout=widgets.Layout(width="32%", min_width="330px", margin="0 6px 8px 0"))


def param_row(cards):
    return widgets.HBox(cards, layout=widgets.Layout(width="100%", justify_content="space-between"))

# ------------------------------------------------------------
# Config runtime e setups
# ------------------------------------------------------------
def runtime_cfg():
    return mesclar_config({
        "setup_name": setup_name_w.value.strip() or "setup_sem_nome",
        "source_mode": source_mode_w.value,
        "camera_index": int(camera_selector_w.value) if str(camera_selector_w.value).isdigit() else 0,
        "video_file": str(video_selector_w.value or ""),
        "video_loop": bool(video_loop_w.value),
        "cap_width": int(cap_width_w.value),
        "cap_height": int(cap_height_w.value),
        "cap_fps": int(cap_fps_w.value),
        "fire_model_path": fire_model_path_w.value.strip(),
        "person_model_path": person_model_path_w.value.strip(),
        "fire_conf": float(fire_conf_w.value),
        "person_conf": float(person_conf_w.value),
        "imgsz_infer": int(imgsz_infer_w.value),
        "detect_every_n_frames": int(detect_every_w.value),
        "max_fire_detections": MAX_FIRE_DETECTIONS_INTERNAL,
        "max_person_detections": MAX_PERSON_DETECTIONS_INTERNAL,
        "min_fire_area_px": MIN_FIRE_AREA_INTERNAL,
        "large_region_area_px": int(large_area_w.value),
        "lower_edge_offset_px": int(lower_offset_w.value),
        "route_step_px": int(route_step_w.value),
        "route_speed_px_frame": int(route_speed_w.value),
        "transition_speed_px_frame": int(transition_speed_w.value),
        "jet_drop_px": int(jet_drop_w.value),
        "jet_pressure_calib": float(pressure_w.value),
        "erase_radius_px": int(erase_radius_w.value),
        "erased_fraction_threshold": float(erased_fraction_w.value),
        "human_safety_radius_px": int(human_radius_w.value),
        "human_memory_seconds": HUMAN_MEMORY_SECONDS,
        "human_safety_use_bbox_distance": True,
        "simulate_extinguish": True,
        "stop_if_no_fire": True,
        "send_to_arduino": True,
        "authorize_real_water": True,
        "arduino_port": arduino_port_w.value.strip() or str(port_selector_w.value or ""),
        "arduino_baud": int(arduino_baud_w.value),
    })


def apply_cfg_to_widgets(cfg):
    cfg = mesclar_config(cfg)
    setup_name_w.value = str(cfg.get("setup_name", DEFAULT_CONFIG["setup_name"]))
    source_mode_w.value = str(cfg.get("source_mode", "camera")) if cfg.get("source_mode", "camera") in ["camera", "video"] else "camera"
    video_loop_w.value = bool(cfg.get("video_loop", True))
    cap_width_w.value = int(cfg.get("cap_width", 640))
    cap_height_w.value = int(cfg.get("cap_height", 480))
    cap_fps_w.value = int(cfg.get("cap_fps", 30))
    fire_model_path_w.value = str(cfg.get("fire_model_path", DEFAULT_FIRE_MODEL_PATH))
    person_model_path_w.value = str(cfg.get("person_model_path", DEFAULT_PERSON_MODEL_NAME))
    fire_conf_w.value = float(cfg.get("fire_conf", 0.10))
    person_conf_w.value = float(cfg.get("person_conf", 0.40))
    imgsz_infer_w.value = int(cfg.get("imgsz_infer", 960))
    detect_every_w.value = int(cfg.get("detect_every_n_frames", 2))
    large_area_w.value = int(cfg.get("large_region_area_px", 2500))
    lower_offset_w.value = int(cfg.get("lower_edge_offset_px", 8))
    route_step_w.value = int(cfg.get("route_step_px", 18))
    route_speed_w.value = int(cfg.get("route_speed_px_frame", max(1, int(DEFAULT_CONFIG["route_speed_px_frame"]) * 2)))
    transition_speed_w.value = int(cfg.get("transition_speed_px_frame", max(1, int(DEFAULT_CONFIG["transition_speed_px_frame"]) * 2)))
    jet_drop_w.value = int(cfg.get("jet_drop_px", 20))
    pressure_w.value = float(cfg.get("jet_pressure_calib", 22.0))
    erase_radius_w.value = int(cfg.get("erase_radius_px", 28))
    erased_fraction_w.value = float(cfg.get("erased_fraction_threshold", 0.45))
    human_radius_w.value = int(cfg.get("human_safety_radius_px", 90))
    arduino_port_w.value = str(cfg.get("arduino_port", ""))
    arduino_baud_w.value = int(cfg.get("arduino_baud", 115200))


def atualizar_lista_setups():
    nomes = listar_nomes_setups()
    atual = setup_dropdown_w.value if setup_dropdown_w.value in nomes else "<nenhum>"
    setup_dropdown_w.options = ["<nenhum>"] + nomes
    setup_dropdown_w.value = atual


def salvar_setup_click(_=None):
    with out_runtime:
        clear_output(wait=True)
        nome = salvar_setup_config(setup_name_w.value, runtime_cfg(), globals().get("SERVO_CALIBRATION", None))
        atualizar_lista_setups()
        setup_dropdown_w.value = nome
        display(HTML(f"Setup salvo: <code>{nome}</code><br>Arquivo: <code>{FIRE_SETUPS_FILE}</code>"))


def carregar_setup_click(_=None):
    with out_runtime:
        clear_output(wait=True)
        nome = setup_dropdown_w.value
        if not nome or nome == "<nenhum>":
            display(HTML("Selecione um setup para carregar."))
            return
        cfg, calib = carregar_setup_config(nome)
        apply_cfg_to_widgets(cfg)
        if calib:
            globals()["SERVO_CALIBRATION"] = calib
            atualizar_status_calibracao()
        display(HTML(f"Setup carregado: <code>{nome}</code>"))

# ------------------------------------------------------------
# Status e painéis
# ------------------------------------------------------------
def set_status(msg, tipo="info"):
    cores = {"info": "#333", "ok": "#116b2f", "warn": "#9a6a00", "erro": "#9b1c1c"}
    status_w.value = f"<b>Status:</b> <span style='color:{cores.get(tipo, '#333')}'>{msg}</span>"


def atualizar_paineis_vazios():
    panel1.value = bgr_to_jpeg_bytes(criar_canvas_vazio(760, 428, "video parado"))
    panel2.value = bgr_to_jpeg_bytes(criar_canvas_vazio(760, 428, "sem rota"))


def resetar_regioes_apagadas(_=None):
    global HIT_MAP, AIM_STATE, ROUTE_EXECUTOR, LAST_FIRE_DETS, LAST_PERSON_DETS, LAST_PLAN, LAST_STATUS_TEXT, FIRE_ACQUIRE_STATE
    HIT_MAP = None
    LAST_FIRE_DETS = []
    LAST_PERSON_DETS = []
    LAST_PLAN = {"route": [], "meta": [], "mode": "sem_rota", "active_fires": [], "erased_fires": []}
    LAST_STATUS_TEXT = "sem chama detectada"
    FIRE_ACQUIRE_STATE = novo_fire_acquire_state() if "novo_fire_acquire_state" in globals() else {"active": False, "start": 0.0, "dets": [], "last_frame": None, "last_base": None}
    AIM_STATE = {"impact": None, "aim": None, "route_index": 0, "route_signature": None, "transitioning": True, "mode": "sem_rota"}
    ROUTE_EXECUTOR = novo_route_executor() if "novo_route_executor" in globals() else {
        "active": False, "plan": None, "freeze_frame": None, "route_index": 0,
        "impact": None, "aim": None, "transitioning": True, "mode": "sem_rota",
        "done": False, "paused_by_human": False,
    }
    resetar_memoria_humanos() if "resetar_memoria_humanos" in globals() else None
    set_status("regioes apagadas reiniciadas", "ok")


def atualizar_last_command(cmd=None, echo=None):
    global LAST_SENT_COMMAND, LAST_SERIAL_ECHO
    if cmd:
        LAST_SENT_COMMAND = cmd
    if echo:
        LAST_SERIAL_ECHO = echo
    last_command_w.value = (
        f"<b>Serial enviado:</b> <code>{LAST_SENT_COMMAND or '-'}</code><br>"
        f"<b>Arduino respondeu:</b> <code>{LAST_SERIAL_ECHO or '-'}</code>"
    )

# ------------------------------------------------------------
# Vídeos e câmeras
# ------------------------------------------------------------
def listar_videos_input():
    exts = ["*.mp4", "*.avi", "*.mov", "*.mkv", "*.webm", "*.m4v"]
    arquivos = []
    for ext in exts:
        arquivos.extend(VIDEO_INPUT_DIR.glob(ext))
    arquivos = sorted(arquivos, key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)
    if not arquivos:
        return [("Nenhum vídeo encontrado", "")]
    return [(p.name, str(p)) for p in arquivos]


def atualizar_dropdown_videos(mostrar_saida=True):
    opts = listar_videos_input()
    video_selector_w.options = opts
    if video_selector_w.value not in [v for _, v in opts]:
        video_selector_w.value = opts[0][1]
    if mostrar_saida:
        with out_runtime:
            clear_output(wait=True)
            display(HTML(
                f"Coloque vídeos em:<br><code>{VIDEO_INPUT_DIR}</code><br><br>"
                f"Vídeos encontrados: <b>{max(0, len(opts) if opts[0][1] else 0)}</b>"
            ))


def solicitar_seek_video(change):
    global VIDEO_SEEK_REQUEST, VIDEO_SEEK_UPDATING
    if VIDEO_SEEK_UPDATING:
        return
    if change.get("name") == "value":
        VIDEO_SEEK_REQUEST["frame"] = int(change.get("new", 0))


video_seek_w.observe(solicitar_seek_video, names="value")


def atualizar_slider_video_por_cap(cap, enabled=True):
    global VIDEO_SEEK_UPDATING
    try:
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    except Exception:
        total = 0
    VIDEO_SEEK_UPDATING = True
    try:
        if enabled and total > 1:
            novo_max = max(1, total - 1)
            if int(video_seek_w.value) > novo_max:
                video_seek_w.value = novo_max
            video_seek_w.max = novo_max
            video_seek_w.disabled = False
            video_seek_w.description = "Avançar vídeo"
        else:
            video_seek_w.disabled = True
            video_seek_w.value = 0
            video_seek_w.max = 1
            video_seek_w.description = "Avançar vídeo"
    finally:
        VIDEO_SEEK_UPDATING = False


def set_slider_video_frame(frame_idx):
    global VIDEO_SEEK_UPDATING
    if video_seek_w.disabled:
        return
    frame_idx = int(np.clip(frame_idx, video_seek_w.min, video_seek_w.max))
    if int(video_seek_w.value) == frame_idx:
        return
    VIDEO_SEEK_UPDATING = True
    try:
        video_seek_w.value = frame_idx
    finally:
        VIDEO_SEEK_UPDATING = False


def obter_nome_camera_linux(dev_path):
    if not shutil.which("v4l2-ctl"):
        return ""
    try:
        result = subprocess.run(["v4l2-ctl", "--device", str(dev_path), "--info"], capture_output=True, text=True, timeout=2)
        texto = result.stdout or result.stderr or ""
        for linha in texto.splitlines():
            if "Card type" in linha:
                return linha.split(":", 1)[-1].strip()
    except Exception:
        pass
    return ""


def testar_camera_opencv(source, width=320, height=240):
    cap = None
    try:
        src = int(source) if isinstance(source, str) and source.isdigit() else source
        cap = cv2.VideoCapture(src, cv2.CAP_V4L2) if platform.system().lower() == "linux" else cv2.VideoCapture(src)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(width))
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(height))
        if not cap.isOpened():
            return False, None
        ok, frame = cap.read()
        if not ok or frame is None:
            return False, None
        h, w = frame.shape[:2]
        return True, (w, h)
    except Exception:
        return False, None
    finally:
        try:
            if cap is not None:
                cap.release()
        except Exception:
            pass


def listar_cameras_robusto(max_indices=10):
    cameras, vistos = [], set()
    for idx in range(max_indices + 1):
        ok, shape = testar_camera_opencv(str(idx))
        if ok:
            label = f"Indice {idx}" + (f" | {shape[0]}x{shape[1]}" if shape else "")
            value = str(idx)
            cameras.append((label, value))
            vistos.add(value)
    for dev in sorted(glob.glob("/dev/video*")):
        if dev in vistos:
            continue
        ok, shape = testar_camera_opencv(dev)
        nome = obter_nome_camera_linux(dev)
        label = dev + (f" | {nome}" if nome else "") + (f" | {shape[0]}x{shape[1]}" if ok and shape else " | detectado, mas nao abriu")
        cameras.append((label, dev))
        vistos.add(dev)
    return cameras or [("Camera 0", "0")]


def atualizar_dropdown_cameras(mostrar_saida=True):
    cameras = listar_cameras_robusto(max_indices=10)
    camera_selector_w.options = cameras
    valores = [v for _, v in cameras]
    camera_selector_w.value = str(DEFAULT_CONFIG["camera_index"]) if str(DEFAULT_CONFIG["camera_index"]) in valores else valores[0]
    if mostrar_saida:
        with out_runtime:
            clear_output(wait=True)
            linhas = ["<b>Cameras detectadas/testadas:</b>"] + [f"<code>{value}</code> — {label}" for label, value in cameras]
            display(HTML("<br>".join(linhas)))


def abrir_fonte_video(cfg):
    if cfg.get("source_mode") == "video":
        path = str(cfg.get("video_file", ""))
        if not path:
            raise RuntimeError(f"Nenhum vídeo selecionado. Coloque vídeos em: {VIDEO_INPUT_DIR}")
        cap = cv2.VideoCapture(path)
        return cap, path
    src = str(camera_selector_w.value or cfg.get("camera_index", 0)).strip()
    src_obj = int(src) if src.isdigit() else src
    cap = cv2.VideoCapture(src_obj, cv2.CAP_V4L2) if platform.system().lower() == "linux" else cv2.VideoCapture(src_obj)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(cfg.get("cap_width", 640)))
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(cfg.get("cap_height", 480)))
    cap.set(cv2.CAP_PROP_FPS, int(cfg.get("cap_fps", 30)))
    return cap, src

# ------------------------------------------------------------
# Serial e Arduino
# ------------------------------------------------------------
def garantir_pyserial_para_runtime():
    global serial, list_ports, SERIAL_AVAILABLE
    try:
        import serial as _serial
        from serial.tools import list_ports as _list_ports
        serial = _serial
        list_ports = _list_ports
        SERIAL_AVAILABLE = True
        return True, "pyserial disponivel"
    except Exception:
        pass
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyserial"], capture_output=True, text=True)
        if result.returncode != 0:
            SERIAL_AVAILABLE = False
            return False, result.stderr or "falha ao instalar pyserial"
        import serial as _serial
        from serial.tools import list_ports as _list_ports
        serial = _serial
        list_ports = _list_ports
        SERIAL_AVAILABLE = True
        return True, "pyserial instalado e disponivel"
    except Exception as e:
        SERIAL_AVAILABLE = False
        return False, str(e)


def listar_portas_seriais_robusto():
    portas, detalhes = [], []
    ok, msg = garantir_pyserial_para_runtime()
    if ok:
        try:
            for p in list_ports.comports():
                device = str(p.device)
                desc = str(getattr(p, "description", "") or "")
                hwid = str(getattr(p, "hwid", "") or "")
                if device and device not in portas:
                    portas.append(device)
                    detalhes.append((device, desc, hwid))
        except Exception as e:
            msg = f"pyserial disponivel, mas falhou ao listar portas: {e}"
    candidatos = []
    candidatos += glob.glob("/dev/ttyACM*")
    candidatos += glob.glob("/dev/ttyUSB*")
    candidatos += glob.glob("/dev/serial/by-id/*")
    candidatos += glob.glob("/dev/cu.usbmodem*")
    candidatos += glob.glob("/dev/cu.usbserial*")
    candidatos += glob.glob("/dev/tty.usbmodem*")
    candidatos += glob.glob("/dev/tty.usbserial*")
    for c in sorted(candidatos):
        try:
            real = os.path.realpath(c)
            label = c if c == real else f"{c} -> {real}"
            device = c
        except Exception:
            label = c
            device = c
        if device not in portas:
            portas.append(device)
            detalhes.append((device, label, "fallback do sistema"))
    porta_digitada = arduino_port_w.value.strip()
    if porta_digitada and porta_digitada not in portas:
        portas.insert(0, porta_digitada)
        detalhes.insert(0, (porta_digitada, "porta digitada", "manual"))
    return portas, detalhes, msg


def atualizar_dropdown_portas(mostrar_saida=True):
    portas, detalhes, msg = listar_portas_seriais_robusto()
    if portas:
        opts = []
        for device, desc, hwid in detalhes:
            texto = device + (f" | {desc}" if desc and desc != device else "")
            opts.append((texto, device))
        port_selector_w.options = opts
        port_selector_w.value = arduino_port_w.value if arduino_port_w.value in portas else portas[0]
        arduino_port_w.value = port_selector_w.value
        if mostrar_saida:
            with out_runtime:
                clear_output(wait=True)
                linhas = ["<b>Portas seriais detectadas:</b>"] + [f"<code>{d}</code> — {desc}" for d, desc, hwid in detalhes]
                display(HTML("<br>".join(linhas)))
        return portas
    port_selector_w.options = [("Nenhuma porta detectada", "")]
    port_selector_w.value = ""
    if mostrar_saida:
        with out_runtime:
            clear_output(wait=True)
            display(HTML(f"Nenhuma porta serial detectada.<br><code>{msg}</code>"))
    return []


def on_port_selector_change(change):
    if change.get("name") == "value":
        arduino_port_w.value = change.get("new") or ""
port_selector_w.observe(on_port_selector_change, names="value")


def ler_serial_disponivel(max_linhas=8):
    if SERIAL_HANDLE is None:
        return ""
    linhas = []
    try:
        for _ in range(max_linhas):
            if getattr(SERIAL_HANDLE, "in_waiting", 0) <= 0:
                break
            line = SERIAL_HANDLE.readline().decode("utf-8", errors="replace").strip()
            if line:
                linhas.append(line)
    except Exception as e:
        linhas.append(f"erro leitura serial: {e}")
    return " | ".join(linhas)


def obter_caminho_sketch_arduino():
    return Path(salvar_sketch_arduino()).resolve()


def abrir_arquivo_no_sistema(path):
    path = Path(path).resolve()
    sistema = platform.system().lower()
    for exe in ["arduino-ide", "arduino"]:
        exe_path = shutil.which(exe)
        if exe_path:
            try:
                subprocess.Popen([exe_path, str(path)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
                return True, f"aberto com {exe_path}"
            except Exception:
                pass
    try:
        if sistema == "linux":
            subprocess.Popen(["xdg-open", str(path)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
            return True, "aberto com xdg-open"
        if sistema == "darwin":
            subprocess.Popen(["open", str(path)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
            return True, "aberto com open"
        if sistema == "windows":
            os.startfile(str(path))
            return True, "aberto com os.startfile"
    except Exception as e:
        return False, str(e)
    return False, "nenhum abridor encontrado"


def abrir_sketch_click(_=None):
    with out_runtime:
        clear_output(wait=True)
        ino = obter_caminho_sketch_arduino()
        ok, msg = abrir_arquivo_no_sistema(ino)
        if ok:
            display(HTML(f"Sketch aberto. Envie para a placa e feche o Serial Monitor antes de conectar.<br><code>{ino}</code>"))
        else:
            display(HTML(f"Abra manualmente:<br><code>{ino}</code><br>{msg}"))


def conectar_serial_click(_=None):
    global SERIAL_HANDLE
    cfg = runtime_cfg()
    porta = cfg.get("arduino_port", "")
    with out_runtime:
        clear_output(wait=True)
        if not porta:
            display(HTML("Selecione uma porta serial antes de conectar."))
            return
        try:
            if SERIAL_HANDLE is not None:
                try:
                    enviar_comando_arduino(SERIAL_HANDLE, None, None, water_on=False, pressure=cfg.get("jet_pressure_calib", 22.0))
                    SERIAL_HANDLE.close()
                except Exception:
                    pass
            SERIAL_HANDLE = abrir_serial_arduino(porta, cfg.get("arduino_baud", 115200))
            SERIAL_HANDLE.write(b"STATUS?\n")
            time.sleep(0.1)
            atualizar_last_command("STATUS?", ler_serial_disponivel())
            display(HTML(f"Arduino conectado em <code>{porta}</code>"))
            set_status(f"Arduino conectado em {porta}", "ok")
        except Exception as e:
            SERIAL_HANDLE = None
            display(HTML(f"Falha ao conectar em <code>{porta}</code><br><code>{type(e).__name__}: {e}</code>"))
            set_status("falha ao conectar Arduino", "erro")


def desconectar_serial_click(_=None):
    global SERIAL_HANDLE
    with out_runtime:
        clear_output(wait=True)
        try:
            if SERIAL_HANDLE is not None:
                enviar_comando_arduino(SERIAL_HANDLE, None, None, water_on=False, pressure=float(pressure_w.value))
                SERIAL_HANDLE.close()
            SERIAL_HANDLE = None
            atualizar_last_command("W0", "desconectado")
            display(HTML("Arduino desconectado."))
            set_status("Arduino desconectado", "info")
        except Exception as e:
            display(HTML(f"Erro ao desconectar:<br><code>{e}</code>"))


def enviar_servos_runtime(pan, tilt, water_on=False):
    if SERIAL_HANDLE is None:
        return
    cmd = enviar_comando_arduino(SERIAL_HANDLE, pan, tilt, water_on=water_on, pressure=float(pressure_w.value))
    echo = ler_serial_disponivel()
    atualizar_last_command(cmd, echo)


def centralizar_servos_click(_=None):
    if SERIAL_HANDLE is None:
        set_status("Arduino nao conectado", "warn")
        return
    try:
        SERIAL_HANDLE.write(b"CENTER\n")
        time.sleep(0.1)
        atualizar_last_command("CENTER", ler_serial_disponivel())
        set_status("servos centralizados", "ok")
    except Exception as e:
        set_status(f"erro ao centralizar: {e}", "erro")


def ping_arduino_click(_=None):
    if SERIAL_HANDLE is None:
        set_status("Arduino nao conectado", "warn")
        return
    try:
        SERIAL_HANDLE.write(b"PING\n")
        SERIAL_HANDLE.write(b"STATUS?\n")
        time.sleep(0.15)
        atualizar_last_command("PING + STATUS?", ler_serial_disponivel())
        set_status("PING enviado", "ok")
    except Exception as e:
        set_status(f"erro ping: {e}", "erro")

# ------------------------------------------------------------
# Calibração dos servos por 3 pontos
# ------------------------------------------------------------
def ponto_calibracao_px(key, frame_shape=None):
    frame_shape = frame_shape or CURRENT_FRAME_SHAPE
    h, w = frame_shape[:2]
    nx, ny = CALIB_POINTS_NORM[key]
    return (int(round(nx * w)), int(round(ny * h)))


def desenhar_marcadores_calibracao(img):
    out = img.copy()
    for key in ["tl", "tr", "bl"]:
        x, y = ponto_calibracao_px(key, out.shape)
        color = (0, 0, 255) if key == ACTIVE_CALIB_POINT else (255, 255, 0)
        cv2.drawMarker(out, (x, y), color, markerType=cv2.MARKER_CROSS, markerSize=28, thickness=2)
        put_label(out, CALIB_LABELS[key], (x + 8, max(18, y - 8)), bg=(70, 70, 0), scale=0.45)
    return out


def atualizar_status_calibracao():
    calib = globals().get("SERVO_CALIBRATION", {})
    if calib.get("enabled"):
        calib_status_w.value = "<b>Calibração:</b> ativa, usando mapeamento afim de 3 pontos"
    else:
        calib_status_w.value = "<b>Calibração:</b> inativa; usando fallback linear"


def set_active_calib_point(key):
    global ACTIVE_CALIB_POINT
    ACTIVE_CALIB_POINT = key
    set_status(f"calibrando ponto {CALIB_LABELS[key]}", "info")
    # Envia imediatamente o valor atual do slider para ajudar no ajuste real.
    if SERIAL_HANDLE is not None:
        enviar_servos_runtime(calib_widgets[key]["pan"].value, calib_widgets[key]["tilt"].value, water_on=False)


def slider_calib_changed(change, key):
    if ACTIVE_CALIB_POINT == key and SERIAL_HANDLE is not None:
        enviar_servos_runtime(calib_widgets[key]["pan"].value, calib_widgets[key]["tilt"].value, water_on=False)


def aplicar_calibracao_click(_=None):
    global SERVO_CALIBRATION
    try:
        frame_shape = CURRENT_FRAME_SHAPE
        image_points = {k: ponto_calibracao_px(k, frame_shape) for k in ["tl", "tr", "bl"]}
        servo_points = {k: (float(calib_widgets[k]["pan"].value), float(calib_widgets[k]["tilt"].value)) for k in ["tl", "tr", "bl"]}
        coef_pan, coef_tilt = calcular_mapeamento_afim_3pontos(image_points, servo_points)
        SERVO_CALIBRATION = {
            "enabled": True,
            "frame_size": [int(frame_shape[1]), int(frame_shape[0])],
            "image_points": {k: list(v) for k, v in image_points.items()},
            "servo_points": {k: list(v) for k, v in servo_points.items()},
            "coef_pan": coef_pan,
            "coef_tilt": coef_tilt,
        }
        atualizar_status_calibracao()
        set_status("calibracao 3 pontos aplicada", "ok")
    except Exception as e:
        set_status(f"erro calibracao: {e}", "erro")


def desativar_calibracao_click(_=None):
    SERVO_CALIBRATION["enabled"] = False
    atualizar_status_calibracao()
    set_status("calibracao desativada", "warn")


def salvar_calibracao_click(_=None):
    SERVO_CALIBRATION_FILE.write_text(json.dumps(SERVO_CALIBRATION, indent=2, ensure_ascii=False), encoding="utf-8")
    set_status("calibracao salva", "ok")


def carregar_calibracao_click(_=None):
    global SERVO_CALIBRATION
    if not SERVO_CALIBRATION_FILE.exists():
        set_status("nenhuma calibracao salva", "warn")
        return
    SERVO_CALIBRATION = json.loads(SERVO_CALIBRATION_FILE.read_text(encoding="utf-8"))
    for k, vals in SERVO_CALIBRATION.get("servo_points", {}).items():
        if k in calib_widgets:
            calib_widgets[k]["pan"].value = float(vals[0])
            calib_widgets[k]["tilt"].value = float(vals[1])
    atualizar_status_calibracao()
    set_status("calibracao carregada", "ok")


# ------------------------------------------------------------
# Helpers da revisão: imagem 2 como base, rota congelada, contorno por cor e baixa latência
# ------------------------------------------------------------
def aplicar_regioes_apagadas_preto(img, hit_map):
    """Aplica apagamento preto opaco. Esta imagem e a base da imagem 2 e da detecção de fogo."""
    if img is None:
        return None
    out = img.copy()
    if hit_map is not None and hit_map.shape[:2] == out.shape[:2]:
        mask = hit_map > 0
        if np.any(mask):
            out[mask] = (0, 0, 0)
    return out


def novo_route_executor():
    return {
        "active": False,
        "plan": None,
        "freeze_frame": None,
        "route_index": 0,
        "impact": None,
        "aim": None,
        "transitioning": True,
        "mode": "sem_rota",
        "done": False,
        "paused_by_human": False,
        "frame_shape": CURRENT_FRAME_SHAPE,
    }


def novo_fire_acquire_state():
    return {"active": False, "start": 0.0, "dets": [], "last_frame": None, "last_base": None}


def primeiro_ponto_rota(plan):
    route = (plan or {}).get("route", [])
    return route[0] if route else None


def criar_executor_para_plan(plan, freeze_frame, frame_shape, start_impact=None):
    route = (plan or {}).get("route", [])
    if not route:
        return novo_route_executor()

    impact0 = start_impact
    if impact0 is not None:
        impact0 = (float(impact0[0]), float(impact0[1]))

    aim0 = None
    if impact0 is not None:
        aim0, _ = calcular_mira_compensada(impact0, runtime_cfg(), frame_shape=frame_shape)

    return {
        "active": True,
        "plan": plan,
        "freeze_frame": freeze_frame.copy() if freeze_frame is not None else None,
        "route_index": 0,
        "impact": impact0,
        "aim": aim0,
        "transitioning": True,
        "mode": plan.get("mode", "rota"),
        "done": False,
        "paused_by_human": False,
        "frame_shape": frame_shape,
    }


def executor_to_aim_state(executor):
    plan = executor.get("plan") or {"route": []}
    return {
        "impact": executor.get("impact"),
        "aim": executor.get("aim"),
        "route_index": int(executor.get("route_index", 0)),
        "route_signature": assinatura_rota(plan.get("route", [])) if plan.get("route") else None,
        "transitioning": bool(executor.get("transitioning", True)),
        "mode": executor.get("mode", "sem_rota"),
        "drop_px": float(runtime_cfg().get("jet_drop_px", 0)),
        "frame_shape": executor.get("frame_shape", CURRENT_FRAME_SHAPE),
    }


def ponto_para_seguranca(executor):
    if executor.get("impact") is not None:
        return executor.get("impact")
    plan = executor.get("plan") or {}
    return primeiro_ponto_rota(plan)


def box_area(box):
    x1, y1, x2, y2 = box
    return max(0, int(x2) - int(x1)) * max(0, int(y2) - int(y1))


def inter_area(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    return max(0, ix2 - ix1) * max(0, iy2 - iy1)


def deve_unir_boxes(a, b, iou_thr=0.12, ioa_thr=0.55):
    ia = inter_area(a, b)
    if ia <= 0:
        return False
    aa = max(1, box_area(a))
    ab = max(1, box_area(b))
    iou = ia / max(1, aa + ab - ia)
    ioa = ia / max(1, min(aa, ab))
    # Une se sobrepoe, ou se uma caixa esta praticamente dentro da outra.
    return iou >= iou_thr or ioa >= ioa_thr


def unir_boxes(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return (min(ax1, bx1), min(ay1, by1), max(ax2, bx2), max(ay2, by2))


def fundir_deteccoes_fogo(dets, frame_shape):
    """Remove sobreposicoes: caixas internas/sobrepostas viram uma caixa unificada."""
    h, w = frame_shape[:2]
    itens = []
    for d in dets or []:
        x1, y1, x2, y2 = clip_box(d.get("box", (0, 0, 1, 1)), w, h)
        if box_area((x1, y1, x2, y2)) <= 0:
            continue
        nd = dict(d)
        nd["box"] = (x1, y1, x2, y2)
        nd["area"] = float(box_area(nd["box"]))
        itens.append(nd)

    changed = True
    while changed:
        changed = False
        usados = [False] * len(itens)
        novos = []
        for i, a in enumerate(itens):
            if usados[i]:
                continue
            cur = dict(a)
            usados[i] = True
            for j in range(i + 1, len(itens)):
                if usados[j]:
                    continue
                b = itens[j]
                if deve_unir_boxes(cur["box"], b["box"]):
                    cur["box"] = unir_boxes(cur["box"], b["box"])
                    cur["conf"] = max(float(cur.get("conf", 0)), float(b.get("conf", 0)))
                    cur["label"] = cur.get("label", b.get("label", "fire"))
                    usados[j] = True
                    changed = True
            cur["center"] = ((cur["box"][0] + cur["box"][2]) / 2.0, (cur["box"][1] + cur["box"][3]) / 2.0)
            cur["area"] = float(box_area(cur["box"]))
            novos.append(cur)
        itens = novos
    return sorted(itens, key=lambda d: (d.get("area", 0), d.get("conf", 0)), reverse=True)


def mascara_chama_hsv_bgr(img_bgr, box):
    """
    Máscara amorfa de chama dentro da bbox YOLO.

    Revisão v18:
    - mantém a região de rota dentro da bbox;
    - concentra a rota em subregiões quentes e brilhantes;
    - inclui o núcleo branco/quase branco da chama com mais tolerância;
    - o branco é aceito quando:
        1) está perto de pixels quentes, OU
        2) pertence a um componente branco brilhante que toca uma região quente,
           considerando uma dilatação maior;
    - isso evita que o zigue-zague desvie do centro branco da chama.
    """
    if img_bgr is None:
        return None

    h, w = img_bgr.shape[:2]
    x1, y1, x2, y2 = clip_box(box, w, h)
    roi = img_bgr[y1:y2 + 1, x1:x2 + 1]

    if roi.size == 0:
        return np.zeros((h, w), dtype=np.uint8)

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    b = roi[:, :, 0].astype(np.int16)
    g = roi[:, :, 1].astype(np.int16)
    r = roi[:, :, 2].astype(np.int16)
    hh = hsv[:, :, 0]
    ss = hsv[:, :, 1]
    vv = hsv[:, :, 2]

    # Limiar adaptativo de brilho dentro da bbox.
    # Um foco de chama pode ter centro branco bem claro e bordas amarelas/laranjas.
    v_p65 = float(np.percentile(vv, 65))
    v_p82 = float(np.percentile(vv, 82))
    bright_thr = int(np.clip(max(135, v_p65), 135, 215))
    very_bright_thr = int(np.clip(max(165, v_p82), 165, 238))

    # Vermelho / laranja / amarelo.
    warm_hsv_1 = (
        (hh >= 0) & (hh <= 60) &
        (ss >= 28) &
        (vv >= bright_thr)
    )
    warm_hsv_2 = (
        (hh >= 165) & (hh <= 180) &
        (ss >= 28) &
        (vv >= bright_thr)
    )

    # Condição BGR para tons de chama, permitindo amarelo claro.
    warm_bgr = (
        (r >= 135) &
        (g >= 70) &
        (vv >= bright_thr) &
        (r >= b + 18) &
        (g >= b * 0.62)
    )

    warm_base = (warm_hsv_1 | warm_hsv_2 | warm_bgr).astype(np.uint8) * 255

    # Centro branco/quase branco da chama.
    # Aqui a condição é menos restritiva que na v17, porque o centro branco real
    # estava ficando fora do zigue-zague.
    white_core_candidate = (
        (vv >= very_bright_thr) &
        (r >= 155) &
        (g >= 145) &
        (b >= 105) &
        ((np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])) <= 125)
    )

    # Branco quente/amarelado muito brilhante, mesmo com baixa saturação.
    white_warm_candidate = (
        (vv >= very_bright_thr) &
        (r >= 160) &
        (g >= 145) &
        (b >= 90) &
        (r >= b - 10) &
        (g >= b - 25)
    )

    white_candidate = (white_core_candidate | white_warm_candidate).astype(np.uint8) * 255

    # Região muito brilhante com tendência quente.
    hot_bright = (
        (vv >= very_bright_thr) &
        (r >= 155) &
        (g >= 110) &
        (r >= b - 5) &
        (g >= b * 0.58)
    ).astype(np.uint8) * 255

    # Aceita branco perto de regiões quentes.
    # A dilatação maior permite incluir o miolo branco quando ele fica separado
    # das bordas quentes por pequenos buracos/estouros de exposição.
    near_warm_small = cv2.dilate(warm_base, np.ones((17, 17), np.uint8), iterations=1) > 0
    near_warm_large = cv2.dilate(warm_base, np.ones((35, 35), np.uint8), iterations=1) > 0

    white_near = ((white_candidate > 0) & (near_warm_small | near_warm_large)).astype(np.uint8) * 255

    # Inclusão por componente: se um componente branco brilhante toca uma região quente
    # após uma dilatação moderada, o componente inteiro entra na máscara.
    white_components = np.zeros_like(white_candidate)
    cnts_white, _ = cv2.findContours(white_candidate, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    warm_touch = cv2.dilate(warm_base, np.ones((41, 41), np.uint8), iterations=1) > 0
    bbox_area = max(1, roi.shape[0] * roi.shape[1])

    for c in cnts_white:
        area = cv2.contourArea(c)
        if area < max(5, int(0.0003 * bbox_area)):
            continue

        tmp = np.zeros_like(white_candidate)
        cv2.drawContours(tmp, [c], -1, 255, -1)

        touches_warm = np.any((tmp > 0) & warm_touch)

        # Se não houver warm_base por saturação/estouro de exposição,
        # ainda aceita componente branco muito brilhante, desde que ele não ocupe
        # quase toda a bbox.
        not_too_large = area < 0.70 * bbox_area
        very_bright_component = np.percentile(vv[tmp > 0], 80) >= very_bright_thr if np.any(tmp > 0) else False

        if touches_warm or (very_bright_component and not_too_large and np.count_nonzero(warm_base) > 0):
            cv2.drawContours(white_components, [c], -1, 255, -1)

    mask_roi = cv2.bitwise_or(warm_base, hot_bright)
    mask_roi = cv2.bitwise_or(mask_roi, white_near)
    mask_roi = cv2.bitwise_or(mask_roi, white_components)

    # Fecha lacunas entre centro branco e bordas quentes,
    # mas sem expandir até ocupar a bbox inteira.
    k3 = np.ones((3, 3), np.uint8)
    k5 = np.ones((5, 5), np.uint8)
    k7 = np.ones((7, 7), np.uint8)
    mask_roi = cv2.morphologyEx(mask_roi, cv2.MORPH_CLOSE, k7, iterations=1)
    mask_roi = cv2.morphologyEx(mask_roi, cv2.MORPH_CLOSE, k5, iterations=1)
    mask_roi = cv2.morphologyEx(mask_roi, cv2.MORPH_OPEN, k3, iterations=1)

    # Mantém componentes relevantes.
    cnts, _ = cv2.findContours(mask_roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean = np.zeros_like(mask_roi)
    min_area = max(10, int(0.0006 * bbox_area))

    warm_or_white_support = cv2.bitwise_or(warm_base, white_components)
    warm_or_white_support = cv2.bitwise_or(warm_or_white_support, white_near)
    warm_or_white_support = cv2.bitwise_or(warm_or_white_support, hot_bright)

    for c in cnts:
        area = cv2.contourArea(c)
        if area < min_area:
            continue

        tmp = np.zeros_like(mask_roi)
        cv2.drawContours(tmp, [c], -1, 255, -1)

        support = np.count_nonzero((tmp > 0) & (warm_or_white_support > 0))
        if support <= 0:
            continue

        # Evita transformar a bbox inteira em fogo por reflexo muito amplo.
        if area > 0.85 * bbox_area and np.count_nonzero(warm_base) < 0.03 * bbox_area:
            continue

        cv2.drawContours(clean, [c], -1, 255, -1)

    # Fallback conservador: volta à máscara, mas nunca à bbox inteira.
    if np.count_nonzero(clean) == 0 and np.count_nonzero(mask_roi) > 0:
        clean = mask_roi

    full = np.zeros((h, w), dtype=np.uint8)
    full[y1:y2 + 1, x1:x2 + 1] = clean
    return full


def contornos_chama_para_deteccao(img_bgr, det):
    """Refina a detecção YOLO usando contornos amorfos de cor/ brilho de chama dentro da bbox."""
    h, w = img_bgr.shape[:2]
    mask = mascara_chama_hsv_bgr(img_bgr, det["box"])
    if mask is None:
        return []
    x1b, y1b, x2b, y2b = clip_box(det["box"], w, h)
    roi_mask = mask[y1b:y2b + 1, x1b:x2b + 1]
    cnts, _ = cv2.findContours(roi_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    refinados = []
    for c in cnts:
        area = float(cv2.contourArea(c))
        if area < 10:
            continue
        c_full = c.copy()
        c_full[:, 0, 0] += x1b
        c_full[:, 0, 1] += y1b
        x, y, ww, hh = cv2.boundingRect(c_full)
        x1, y1, x2, y2 = clip_box((x, y, x + ww, y + hh), w, h)
        if box_area((x1, y1, x2, y2)) <= 0:
            continue
        M = cv2.moments(c_full)
        if abs(M["m00"]) > 1e-6:
            cx = float(M["m10"] / M["m00"])
            cy = float(M["m01"] / M["m00"])
        else:
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        local_mask = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(local_mask, [c_full], -1, 255, -1)
        nd = dict(det)
        nd.update({
            "box": (x1, y1, x2, y2),
            "full_box": (x1b, y1b, x2b, y2b),
            "area": area,
            "center": (cx, cy),
            "contour": c_full,
            "mask": local_mask,
            "raw_box": det.get("box"),
        })
        refinados.append(nd)
    return refinados


def refinar_fogos_por_cor_e_contorno(frame_base, dets, frame_shape):
    """
    Refina as detecções YOLO para a imagem 2.

    Revisão v17:
    - une bboxes YOLO sobrepostas/internas;
    - dentro delas, cria uma máscara amorfa por brilho + cor de chama;
    - a rota é construída somente nessa subregião amorfa;
    - não usa a bbox inteira como região de planejamento quando a máscara falha.
    """
    if frame_base is None:
        return []

    h, w = frame_shape[:2]
    unidos = fundir_deteccoes_fogo(dets, frame_shape)

    union_mask = np.zeros((h, w), dtype=np.uint8)
    fonte_por_box = []

    for d in unidos:
        m = mascara_chama_hsv_bgr(frame_base, d["box"])
        if m is None or np.count_nonzero(m) == 0:
            continue

        union_mask = cv2.bitwise_or(union_mask, m)
        fonte_por_box.append(d)

    if np.count_nonzero(union_mask) == 0:
        # Sem subregião confiável de chama: não planeja sobre a bbox inteira.
        return []

    # Fecha pequenas lacunas sem transformar a região em retângulo.
    union_mask = cv2.morphologyEx(union_mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=1)

    cnts, _ = cv2.findContours(union_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    refinados = []
    for c in cnts:
        area = float(cv2.contourArea(c))
        if area < 10:
            continue

        x, y, ww, hh = cv2.boundingRect(c)
        x1, y1, x2, y2 = clip_box((x, y, x + ww, y + hh), w, h)

        if box_area((x1, y1, x2, y2)) <= 0:
            continue

        local_mask = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(local_mask, [c], -1, 255, -1)

        # Confiança da região: maior confiança da bbox YOLO que contém/intersecta o contorno.
        conf = 0.0
        label = "fire"
        raw_box = (x1, y1, x2, y2)

        for d in unidos:
            if inter_area((x1, y1, x2, y2), d["box"]) > 0:
                conf = max(conf, float(d.get("conf", 0.0)))
                label = str(d.get("label", label))
                raw_box = d.get("box", raw_box)

        M = cv2.moments(c)
        if abs(M["m00"]) > 1e-6:
            cx = float(M["m10"] / M["m00"])
            cy = float(M["m01"] / M["m00"])
        else:
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0

        refinados.append({
            "box": (x1, y1, x2, y2),
            "full_box": raw_box,
            "raw_box": raw_box,
            "area": area,
            "center": (cx, cy),
            "contour": c,
            "mask": local_mask,
            "label": label,
            "conf": conf,
        })

    # Não funde por retângulo aqui, para não voltar a transformar regiões distintas em bbox.
    # Ordena por área para a prioridade da maior região funcionar diretamente.
    refinados = sorted(refinados, key=lambda d: (float(d.get("area", 0)), float(d.get("conf", 0))), reverse=True)
    return refinados


def mascara_uniao_fogos(fires, frame_shape):
    h, w = frame_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    for f in fires or []:
        if isinstance(f.get("mask"), np.ndarray) and f["mask"].shape[:2] == (h, w):
            mask = cv2.bitwise_or(mask, (f["mask"] > 0).astype(np.uint8))
        elif f.get("contour") is not None:
            cv2.drawContours(mask, [f["contour"]], -1, 1, -1)
        else:
            x1, y1, x2, y2 = clip_box(f.get("box", (0, 0, 1, 1)), w, h)
            mask[y1:y2 + 1, x1:x2 + 1] = 1
    return mask


def fire_erased_fraction_contorno(fire_det, hit_map):
    if hit_map is None:
        return 0.0
    h, w = hit_map.shape[:2]
    mask = None
    if isinstance(fire_det.get("mask"), np.ndarray) and fire_det["mask"].shape[:2] == (h, w):
        mask = fire_det["mask"] > 0
    elif fire_det.get("contour") is not None:
        m = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(m, [fire_det["contour"]], -1, 1, -1)
        mask = m > 0
    else:
        x1, y1, x2, y2 = clip_box(fire_det["box"], w, h)
        mask = np.zeros((h, w), dtype=bool)
        mask[y1:y2 + 1, x1:x2 + 1] = True
    total = int(np.count_nonzero(mask))
    if total <= 0:
        return 0.0
    return float(np.count_nonzero((hit_map > 0) & mask) / total)


def filtrar_fogos_nao_apagados_contorno(fires, hit_map, threshold=0.45):
    ativos, apagados = [], []
    for f in fires:
        frac = fire_erased_fraction_contorno(f, hit_map)
        item = dict(f)
        item["erased_fraction"] = frac
        if frac >= float(threshold):
            apagados.append(item)
        else:
            ativos.append(item)
    return ativos, apagados


def mascara_bool_fogo(fire_det, frame_shape):
    h, w = frame_shape[:2]

    if isinstance(fire_det.get("mask"), np.ndarray) and fire_det["mask"].shape[:2] == (h, w):
        return fire_det["mask"] > 0

    if fire_det.get("contour") is not None:
        m = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(m, [fire_det["contour"]], -1, 1, -1)
        return m > 0

    # Fallback conservador: usa um pequeno núcleo elíptico central,
    # nunca a bbox inteira. Isso evita zigue-zague em regiões que não são chama.
    x1, y1, x2, y2 = clip_box(fire_det.get("box", (0, 0, 1, 1)), w, h)
    cx = int((x1 + x2) / 2)
    cy = int((y1 + y2) / 2)
    rx = max(3, int((x2 - x1) * 0.22))
    ry = max(3, int((y2 - y1) * 0.22))
    m = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(m, (cx, cy), (rx, ry), 0, 0, 360, 1, -1)
    return m > 0


def pontos_linha_maior_eixo_contorno(fire_det, cfg, frame_shape):
    """
    Gera uma rota de varredura dentro do contorno amorfo da chama.

    Revisão v15:
    - a rota não apaga a região instantaneamente;
    - ela cobre a máscara da chama por linhas de varredura em zigue-zague;
    - o espaçamento entre sequências do zigue-zague é 2x o raio de atuação;
    - o espaçamento entre pontos dentro de cada sequência usa route_step_px;
    - se a chama for mais larga, as sequências são horizontais;
    - se a chama for mais alta/delgada, as sequências são verticais;
    - os pontos ficam sempre dentro da máscara refinada por cor/brilho.
    """
    h, w = frame_shape[:2]

    point_step = max(3, int(cfg.get("route_step_px", 18)))
    erase_radius = max(1, int(cfg.get("erase_radius_px", 28)))
    sequence_spacing = max(3, int(2 * erase_radius))

    mask = mascara_bool_fogo(fire_det, frame_shape)

    # Remove pontos já apagados para evitar passar a mira onde o jato já atuou.
    if HIT_MAP is not None and HIT_MAP.shape[:2] == mask.shape[:2]:
        mask = mask & ~(HIT_MAP > 0)

    ys_all, xs_all = np.where(mask)
    if len(xs_all) == 0:
        return [ponto_centroide_contorno(fire_det, frame_shape)]

    x_min, x_max = int(xs_all.min()), int(xs_all.max())
    y_min, y_max = int(ys_all.min()), int(ys_all.max())
    width = max(1, x_max - x_min + 1)
    height = max(1, y_max - y_min + 1)

    pts = []

    if height > width:
        # Chama alta/delgada: varre por colunas.
        # O espaçamento entre colunas é 2x o raio de atuação.
        xs_scan = list(range(x_min, x_max + 1, sequence_spacing))
        if x_max not in xs_scan:
            xs_scan.append(x_max)

        reverse = False

        for xx in xs_scan:
            x0 = max(x_min, xx - max(1, sequence_spacing // 2))
            x1 = min(x_max, xx + max(1, sequence_spacing // 2))
            sub = mask[y_min:y_max + 1, x0:x1 + 1]
            rr, cc = np.where(sub)
            if len(rr) == 0:
                continue

            ys = y_min + rr
            xs = x0 + cc

            y_bins = list(range(int(ys.min()), int(ys.max()) + 1, point_step))
            if int(ys.max()) not in y_bins:
                y_bins.append(int(ys.max()))

            col_pts = []
            for yy in y_bins:
                band = np.abs(ys - yy) <= max(1, point_step // 2)
                if not np.any(band):
                    continue

                px = int(np.clip(round(np.mean(xs[band])), 0, w - 1))
                py = int(np.clip(round(np.mean(ys[band])), 0, h - 1))
                col_pts.append((px, py))

            if reverse:
                col_pts = list(reversed(col_pts))

            pts.extend(col_pts)
            reverse = not reverse

    else:
        # Chama larga: varre por linhas.
        # O espaçamento entre linhas é 2x o raio de atuação.
        ys_scan = list(range(y_min, y_max + 1, sequence_spacing))
        if y_max not in ys_scan:
            ys_scan.append(y_max)

        reverse = False

        for yy in ys_scan:
            y0 = max(y_min, yy - max(1, sequence_spacing // 2))
            y1 = min(y_max, yy + max(1, sequence_spacing // 2))
            sub = mask[y0:y1 + 1, x_min:x_max + 1]
            rr, cc = np.where(sub)
            if len(rr) == 0:
                continue

            xs = x_min + cc
            ys = y0 + rr

            x_bins = list(range(int(xs.min()), int(xs.max()) + 1, point_step))
            if int(xs.max()) not in x_bins:
                x_bins.append(int(xs.max()))

            row_pts = []
            for xx in x_bins:
                band = np.abs(xs - xx) <= max(1, point_step // 2)
                if not np.any(band):
                    continue

                px = int(np.clip(round(np.mean(xs[band])), 0, w - 1))
                py = int(np.clip(round(np.mean(ys[band])), 0, h - 1))
                row_pts.append((px, py))

            if reverse:
                row_pts = list(reversed(row_pts))

            pts.extend(row_pts)
            reverse = not reverse

    if not pts:
        pts = [ponto_centroide_contorno(fire_det, frame_shape)]

    # Remove pontos muito próximos para reduzir tremedeira e comandos redundantes.
    cleaned = []
    min_dist = max(2, point_step * 0.35)
    for p in pts:
        if not cleaned or np.hypot(p[0] - cleaned[-1][0], p[1] - cleaned[-1][1]) >= min_dist:
            cleaned.append(p)

    return cleaned or pts


def pontos_borda_inferior_contorno(fire_det, cfg, frame_shape):
    return pontos_linha_maior_eixo_contorno(fire_det, cfg, frame_shape)

def ponto_centroide_contorno(fire_det, frame_shape):
    h, w = frame_shape[:2]
    if fire_det.get("contour") is not None:
        M = cv2.moments(fire_det["contour"])
        if abs(M["m00"]) > 1e-6:
            return (int(np.clip(round(M["m10"] / M["m00"]), 0, w - 1)), int(np.clip(round(M["m01"] / M["m00"]), 0, h - 1)))
    return ponto_centroide(fire_det, frame_shape)


def ordenar_pontos_por_proximidade(pts, current_point=None):
    if not pts:
        return pts
    if current_point is None:
        return pts
    cx, cy = float(current_point[0]), float(current_point[1])
    d0 = np.hypot(pts[0][0] - cx, pts[0][1] - cy)
    d1 = np.hypot(pts[-1][0] - cx, pts[-1][1] - cy)
    return list(reversed(pts)) if d1 < d0 else pts


def escolher_fogo_mais_proximo(fires, frame_shape, current_point=None):
    """
    Escolhe a região de fogo prioritária.

    Revisão v16:
    - se houver mais de uma região de fogo ativa, a prioridade é sempre da maior área;
    - a posição atual do jato deixa de ser critério principal;
    - distância fica apenas como desempate.
    """
    if not fires:
        return None

    h, w = frame_shape[:2]
    if current_point is None:
        current_point = (w * 0.5, h - 8)

    cx, cy = float(current_point[0]), float(current_point[1])

    def score(f):
        area = float(f.get("area", 0))
        c = ponto_centroide_contorno(f, frame_shape)
        dist = float(np.hypot(c[0] - cx, c[1] - cy))
        # Maior área primeiro; distância apenas desempata.
        return (-area, dist)

    return sorted(fires, key=score)[0]


def planejar_rota_combate(fires, hit_map, cfg, frame_shape, current_point=None):
    """
    Planeja uma rota para uma única região por vez.

    Revisão v13:
    - escolhe apenas uma região ativa, a mais próxima da posição atual do jato;
    - usa contorno amorfo filtrado por cor/brilho;
    - para regiões grandes, gera varredura em zigue-zague dentro do contorno;
    - para regiões pequenas, usa centróide repetido por alguns ciclos;
    - evita regiões já apagadas usando o hit_map.
    """
    large_thr = int(cfg.get("large_region_area_px", 2500))
    erase_thr = float(cfg.get("erased_fraction_threshold", 0.45))
    fires_active, fires_erased = filtrar_fogos_nao_apagados_contorno(fires, hit_map, erase_thr)

    target = escolher_fogo_mais_proximo(fires_active, frame_shape, current_point=current_point)
    route, meta = [], []
    mode = "sem_rota"

    if target is not None:
        if float(target.get("area", 0)) >= large_thr:
            pts = pontos_linha_maior_eixo_contorno(target, cfg, frame_shape)
            pts = ordenar_pontos_por_proximidade(pts, current_point=current_point)
            for p in pts:
                route.append(p)
                meta.append({"mode": "eixo_contorno_otimizado", "fire": target})
            mode = "eixo_contorno_otimizado"
        else:
            c = ponto_centroide_contorno(target, frame_shape)
            hold = max(1, int(cfg.get("centroid_hold_frames", 5)))
            for _ in range(hold):
                route.append(c)
                meta.append({"mode": "centroide_contorno", "fire": target})
            mode = "centroide_contorno"

    fire_mask = mascara_uniao_fogos([target] if target is not None else [], frame_shape)
    return {
        "route": route,
        "meta": meta,
        "mode": mode if route else "sem_rota",
        "active_fires": [target] if target is not None else [],
        "erased_fires": fires_erased,
        "all_active_fires": fires_active,
        "large_count": 1 if target is not None and float(target.get("area", 0)) >= large_thr else 0,
        "small_count": 1 if target is not None and float(target.get("area", 0)) < large_thr else 0,
        "fire_mask": fire_mask,
    }


def registrar_hit_apagamento_fogo(hit_map, impact, radius, plan=None):
    """
    Simula a atuação do jato, sem apagar a região inteira instantaneamente.

    A cada ciclo, apenas um círculo ao redor do ponto de impacto é pintado de preto.
    Quando existe máscara refinada da chama, esse círculo é limitado à região de fogo
    contida na bbox/contorno. Assim, quem "apaga" é o jato ao percorrer a rota.
    """
    if hit_map is None or impact is None:
        return hit_map

    h, w = hit_map.shape[:2]
    x, y = int(round(impact[0])), int(round(impact[1]))

    circ = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(circ, (x, y), int(radius), 1, -1)

    fire_mask = None
    if isinstance((plan or {}).get("fire_mask"), np.ndarray) and plan["fire_mask"].shape[:2] == (h, w):
        fire_mask = plan["fire_mask"].astype(np.uint8)
        fire_mask = cv2.dilate(fire_mask, np.ones((5, 5), np.uint8), iterations=1)

    if fire_mask is not None and np.any(fire_mask):
        hit_map[((circ > 0) & (fire_mask > 0))] = 1
    else:
        hit_map[circ > 0] = 1

    return hit_map


def atualizar_executor_rota(executor, cfg, blocked_by_human=False):
    """
    Move a mira apenas quando a rota esta ativa.
    Se houver humano perto do impacto, a rota permanece congelada no ponto atual.
    Retorna: executor, moved, finished_now
    """
    if not executor.get("active", False):
        return executor, False, False

    plan = executor.get("plan") or {}
    route = plan.get("route", [])
    meta = plan.get("meta", [])
    if not route:
        executor.update(novo_route_executor())
        return executor, False, True

    if blocked_by_human:
        executor["paused_by_human"] = True
        return executor, False, False

    executor["paused_by_human"] = False
    idx = max(0, min(int(executor.get("route_index", 0)), len(route) - 1))
    target = route[idx]
    mode = meta[idx].get("mode", "rota") if idx < len(meta) else "rota"

    if executor.get("transitioning", True) or "centroide" in mode:
        speed = float(cfg.get("transition_speed_px_frame", 35))
    else:
        speed = float(cfg.get("route_speed_px_frame", 14))

    new_impact, reached = move_point_towards(executor.get("impact"), target, speed)
    executor["impact"] = new_impact
    executor["mode"] = mode
    executor["frame_shape"] = executor.get("frame_shape", CURRENT_FRAME_SHAPE)
    aim, drop = calcular_mira_compensada(new_impact, cfg, frame_shape=executor.get("frame_shape"))
    executor["aim"] = aim
    executor["drop_px"] = drop

    finished_now = False
    if reached:
        if idx < len(route) - 1:
            executor["route_index"] = idx + 1
            executor["transitioning"] = False
        else:
            executor["active"] = False
            executor["done"] = True
            executor["transitioning"] = True
            finished_now = True
    return executor, True, finished_now


def desenhar_jato_do_bico(out, impact, cfg):
    """Ilustração visual do arco do jato saindo do bico até o ponto de impacto."""
    if impact is None:
        return out
    h, w = out.shape[:2]
    nozzle = (int(w * 0.50), h - 8)
    ix, iy = int(round(impact[0])), int(round(impact[1]))
    drop = float(cfg.get("jet_drop_px", 20))
    arc = float(np.clip(35 + abs(nozzle[1] - iy) * 0.10 + abs(nozzle[0] - ix) * 0.08 + drop * 0.8, 25, 180))
    pts = []
    for t in np.linspace(0, 1, 36):
        x = (1 - t) * nozzle[0] + t * ix
        y = (1 - t) * nozzle[1] + t * iy - arc * math.sin(math.pi * t)
        pts.append((int(np.clip(round(x), 0, w - 1)), int(np.clip(round(y), 0, h - 1))))
    cv2.polylines(out, [np.array(pts, dtype=np.int32).reshape(-1, 1, 2)], False, (255, 220, 80), 2)
    cv2.circle(out, nozzle, 5, (255, 180, 40), -1)
    return out


def desenhar_contornos_fogo(out, fires):
    """
    Desenha somente contornos amorfos na imagem 2.
    Não desenha bbox retangular aqui; as caixas ficam apenas na imagem 1.
    """
    for f in fires or []:
        drew = False

        if isinstance(f.get("mask"), np.ndarray) and f["mask"].shape[:2] == out.shape[:2]:
            m = (f["mask"] > 0).astype(np.uint8) * 255
            cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if cnts:
                cv2.drawContours(out, cnts, -1, (0, 140, 255), 2)
                drew = True

        if (not drew) and f.get("contour") is not None:
            cv2.drawContours(out, [f["contour"]], -1, (0, 140, 255), 2)
            drew = True

        # Fallback sem caixa: se não houver contorno/máscara, desenha um pequeno círculo no centróide.
        if not drew:
            try:
                c = ponto_centroide_contorno(f, out.shape)
                cv2.circle(out, (int(c[0]), int(c[1])), 5, (0, 140, 255), 2)
            except Exception:
                pass


def desenhar_painel_combate_congelado(frame, plan, aim_state, persons, hit_map, cfg, blocked=False, frozen=False, acquiring=False):
    """
    Desenha a imagem 2.
    - Apagamento em preto opaco.
    - Contornos reais da chama.
    - Rota planejada em magenta.
    - Jato visual saindo do bico.
    - Mira verde e impacto vermelho.
    """
    base = aplicar_regioes_apagadas_preto(frame, hit_map)
    out = base.copy()

    plan = plan or {"route": [], "meta": [], "mode": "sem_rota", "active_fires": [], "erased_fires": []}
    route = plan.get("route", [])
    active = plan.get("active_fires", [])
    erased = plan.get("erased_fires", [])
    impact = aim_state.get("impact")
    aim = aim_state.get("aim")
    idx = int(aim_state.get("route_index", 0))

    desenhar_contornos_fogo(out, active)
    for f in erased:
        if f.get("contour") is not None:
            cv2.drawContours(out, [f["contour"]], -1, (80, 80, 80), 1)
        else:
            x1, y1, x2, y2 = f["box"]
            cv2.rectangle(out, (x1, y1), (x2, y2), (80, 80, 80), 1)

    if route:
        pts = np.array([(int(x), int(y)) for x, y in route], dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(out, [pts], False, (255, 0, 255), 2)
        for k, p in enumerate(route):
            color = (255, 0, 255) if k >= idx else (70, 70, 70)
            cv2.circle(out, (int(p[0]), int(p[1])), 2, color, -1)

    out = desenhar_jato_do_bico(out, impact, cfg)

    if aim is not None:
        ax, ay = int(round(aim[0])), int(round(aim[1]))
        cv2.circle(out, (ax, ay), 5, (0, 255, 0), -1)  # mira verde

    if impact is not None:
        ix, iy = int(round(impact[0])), int(round(impact[1]))
        cv2.circle(out, (ix, iy), 5, (0, 0, 255), -1)  # impacto vermelho
        cv2.circle(out, (ix, iy), int(cfg.get("erase_radius_px", 28)), (0, 0, 255), 1)

    mode = plan.get("mode", "sem_rota")
    if acquiring:
        status = "fogo detectado; congelando imagem 2"
    else:
        status = f"rota: {mode} | {'CONGELADA' if frozen else 'tempo real'} | alvo={len(active)} apagadas={len(erased)}"
    if blocked:
        status += " | BLOQUEADO humano"
    put_label(out, status, (10, 28), bg=(0, 0, 150) if blocked else (75, 45, 110), scale=0.58, thickness=2)
    put_label(out, "mira=verde | impacto=vermelho | preto=apagado | contorno=subregiao_chama", (10, 58), bg=(45, 45, 45), scale=0.50)
    return out


class LatestFrameSource:
    """Leitor com frame mais recente para camera, evitando fila atrasada depois do YOLO."""
    def __init__(self, cap, source_mode="camera", loop_video=False):
        self.cap = cap
        self.source_mode = source_mode
        self.loop_video = loop_video
        self.running = False
        self.lock = threading.Lock()
        self.latest = None
        self.ok = False
        self.thread = None
        if source_mode == "camera":
            self.running = True
            self.thread = threading.Thread(target=self._reader_loop, daemon=True)
            self.thread.start()

    def _reader_loop(self):
        while self.running:
            ok, frame = self.cap.read()
            if ok and frame is not None:
                with self.lock:
                    self.latest = frame
                    self.ok = True
            else:
                time.sleep(0.005)

    def read(self):
        if self.source_mode == "camera":
            # Aguarda brevemente o primeiro frame.
            for _ in range(30):
                with self.lock:
                    if self.latest is not None:
                        return True, self.latest.copy()
                time.sleep(0.01)
            return False, None
        ok, frame = self.cap.read()
        if (not ok or frame is None) and self.loop_video:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ok, frame = self.cap.read()
        return ok, frame

    def seek(self, frame_idx):
        if self.source_mode == "camera":
            return False
        try:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, int(max(0, frame_idx)))
            return True
        except Exception:
            return False

    def current_frame_index(self):
        try:
            return int(self.cap.get(cv2.CAP_PROP_POS_FRAMES))
        except Exception:
            return 0

    def stop(self):
        self.running = False
        if self.thread is not None:
            try:
                self.thread.join(timeout=0.5)
            except Exception:
                pass
        try:
            self.cap.release()
        except Exception:
            pass
# ------------------------------------------------------------
# Memória de humanos para segurança
# ------------------------------------------------------------
def atualizar_memoria_humanos(person_dets, now=None):
    """Mantém a última bbox/centróide de humanos por 3 s."""
    global HUMAN_MEMORY
    now = time.time() if now is None else float(now)
    if person_dets:
        mem = []
        for d in person_dets:
            nd = dict(d)
            if "center" not in nd and "box" in nd:
                x1, y1, x2, y2 = nd["box"]
                nd["center"] = ((x1 + x2) / 2.0, (y1 + y2) / 2.0)
            nd["memory_age"] = 0.0
            nd["from_memory"] = False
            mem.append(nd)
        HUMAN_MEMORY = {"timestamp": now, "persons": mem}
        return mem

    age = now - float(HUMAN_MEMORY.get("timestamp", 0.0))
    if HUMAN_MEMORY.get("persons") and age <= HUMAN_MEMORY_SECONDS:
        mem = []
        for d in HUMAN_MEMORY.get("persons", []):
            nd = dict(d)
            nd["memory_age"] = age
            nd["from_memory"] = True
            mem.append(nd)
        return mem
    return []


def resetar_memoria_humanos():
    global HUMAN_MEMORY, LAST_SAFETY_PERSON_DETS
    HUMAN_MEMORY = {"timestamp": 0.0, "persons": []}
    LAST_SAFETY_PERSON_DETS = []


# ------------------------------------------------------------
# Loop principal
# ------------------------------------------------------------
async def monitor_loop():
    global MONITOR_RUNNING, HIT_MAP, LAST_FIRE_DETS, LAST_PERSON_DETS, LAST_SAFETY_PERSON_DETS
    global LAST_DETECT_FRAME, FRAME_COUNTER, CURRENT_FRAME_SHAPE
    global AIM_STATE, ROUTE_EXECUTOR, LAST_PLAN, LAST_BLOCKED_BY_HUMAN, LAST_STATUS_TEXT, FIRE_ACQUIRE_STATE
    global VIDEO_SEEK_REQUEST

    cfg = runtime_cfg()
    set_status("carregando modelos YOLO...", "info")
    try:
        fire_model, person_model = carregar_modelos_runtime(cfg)
    except Exception as e:
        set_status("erro ao carregar modelos", "erro")
        with out_runtime:
            clear_output(wait=True)
            display(HTML(f"Erro ao carregar YOLO:<br><code>{type(e).__name__}: {e}</code>"))
        MONITOR_RUNNING = False
        return

    fire_class_ids = encontrar_classes_por_palavras(fire_model, ["fire", "flame", "chama", "fogo"])
    person_class_ids = encontrar_classes_por_palavras(person_model, ["person", "pessoa", "humano"])
    if not person_class_ids:
        person_class_ids = [0]

    try:
        cap, fonte = abrir_fonte_video(cfg)
    except Exception as e:
        set_status("erro ao abrir fonte", "erro")
        with out_runtime:
            clear_output(wait=True)
            display(HTML(f"{e}"))
        MONITOR_RUNNING = False
        return

    if not cap.isOpened():
        set_status(f"nao foi possivel abrir fonte {fonte}", "erro")
        MONITOR_RUNNING = False
        return

    try:
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    except Exception:
        pass

    atualizar_slider_video_por_cap(cap, enabled=(cfg.get("source_mode") == "video"))
    grabber = LatestFrameSource(cap, source_mode=cfg.get("source_mode", "camera"), loop_video=bool(cfg.get("video_loop", True)))
    ok_teste, frame_teste = grabber.read()
    if not ok_teste or frame_teste is None:
        grabber.stop()
        set_status(f"fonte abriu, mas nao entregou frame: {fonte}", "erro")
        MONITOR_RUNNING = False
        return

    set_status(f"executando: {fonte}", "ok")
    LAST_FIRE_DETS = []
    LAST_PERSON_DETS = []
    LAST_SAFETY_PERSON_DETS = []
    resetar_memoria_humanos()
    LAST_DETECT_FRAME = -999
    FRAME_COUNTER = 0
    LAST_PLAN = {"route": [], "meta": [], "mode": "sem_rota", "active_fires": [], "erased_fires": []}
    ROUTE_EXECUTOR = novo_route_executor()
    FIRE_ACQUIRE_STATE = novo_fire_acquire_state()
    AIM_STATE = executor_to_aim_state(ROUTE_EXECUTOR)
    LAST_BLOCKED_BY_HUMAN = False
    LAST_STATUS_TEXT = "sem chama detectada"

    try:
        while MONITOR_RUNNING:
            cfg = runtime_cfg()
            ok, frame = grabber.read()
            if not ok or frame is None:
                set_status("fim do video ou falha ao ler frame", "warn")
                MONITOR_RUNNING = False
                break

            if cfg.get("source_mode") == "video":
                req = VIDEO_SEEK_REQUEST.get("frame")
                if req is not None:
                    grabber.seek(req)
                    VIDEO_SEEK_REQUEST["frame"] = None
                    ROUTE_EXECUTOR = novo_route_executor()
                    LAST_PLAN = {"route": [], "meta": [], "mode": "sem_rota", "active_fires": [], "erased_fires": []}
                    LAST_FIRE_DETS = []
                    # Lê imediatamente o frame solicitado para atualizar os painéis.
                    ok_seek, frame_seek = grabber.read()
                    if ok_seek and frame_seek is not None:
                        frame = frame_seek
                    set_status(f"vídeo avançado para frame {req}; rota reiniciada", "info")
                if FRAME_COUNTER % 3 == 0:
                    set_slider_video_frame(grabber.current_frame_index())

            CURRENT_FRAME_SHAPE = frame.shape
            FRAME_COUNTER += 1
            h, w = frame.shape[:2]
            if HIT_MAP is None or HIT_MAP.shape[:2] != (h, w):
                HIT_MAP = criar_mapa_hits(frame.shape)

            detect_every = max(1, int(cfg.get("detect_every_n_frames", 6)))
            do_control_cycle = (FRAME_COUNTER - LAST_DETECT_FRAME) >= detect_every

            # Imagem 2: tempo real enquanto nao ha rota. Congela no frame usado para a rota.
            if ROUTE_EXECUTOR.get("active", False) and ROUTE_EXECUTOR.get("freeze_frame") is not None:
                combat_base_frame = ROUTE_EXECUTOR["freeze_frame"]
                panel2_frozen = True
            else:
                combat_base_frame = aplicar_regioes_apagadas_preto(frame, HIT_MAP)
                panel2_frozen = False

            acquiring = bool(FIRE_ACQUIRE_STATE.get("active", False))

            if do_control_cycle:
                # Segurança continua no ciclo de controle, inclusive com a imagem 2 congelada.
                # Se precisar de mais fluidez, aumente o slider "Ciclo controle".
                LAST_PERSON_DETS = detectar_yolo(
                    frame,
                    person_model,
                    conf=cfg.get("person_conf", 0.40),
                    imgsz=cfg.get("imgsz_infer", 960),
                    classes=person_class_ids,
                    max_det=8,
                )
                LAST_SAFETY_PERSON_DETS = atualizar_memoria_humanos(LAST_PERSON_DETS)

                finished_now = False

                if ROUTE_EXECUTOR.get("active", False):
                    # Durante rota ativa, congela todos os processos internos de fogo/planejamento.
                    # Apenas segurança humana, movimento da mira e serial continuam.
                    safety_point = ponto_para_seguranca(ROUTE_EXECUTOR)
                    LAST_BLOCKED_BY_HUMAN, near_person, near_dist = humano_perto_do_impacto(
                        LAST_SAFETY_PERSON_DETS,
                        safety_point,
                        cfg.get("human_safety_radius_px", 90),
                        usar_bbox=True,
                    )

                    ROUTE_EXECUTOR, moved, finished_now = atualizar_executor_rota(
                        ROUTE_EXECUTOR,
                        cfg,
                        blocked_by_human=LAST_BLOCKED_BY_HUMAN,
                    )

                    impact_now = ROUTE_EXECUTOR.get("impact")
                    if moved and impact_now is not None and not LAST_BLOCKED_BY_HUMAN:
                        HIT_MAP = registrar_hit_apagamento_fogo(
                            HIT_MAP,
                            impact_now,
                            cfg.get("erase_radius_px", 28),
                            plan=ROUTE_EXECUTOR.get("plan"),
                        )

                    AIM_STATE = executor_to_aim_state(ROUTE_EXECUTOR)
                    impact = AIM_STATE.get("impact")
                    aim = AIM_STATE.get("aim")
                    water_on = bool(
                        ROUTE_EXECUTOR.get("active", False)
                        and impact is not None
                        and not LAST_BLOCKED_BY_HUMAN
                        and cfg.get("authorize_real_water", True)
                    )

                    if LAST_BLOCKED_BY_HUMAN:
                        LAST_STATUS_TEXT = "JATO BLOQUEADO: humano perto do impacto"
                    elif ROUTE_EXECUTOR.get("active", False):
                        LAST_STATUS_TEXT = "ROTA CONGELADA: executando ate concluir" + (" / agua ON" if water_on else " / agua OFF")
                    elif finished_now:
                        LAST_STATUS_TEXT = "rota concluida; imagem 2 voltou ao tempo real"
                        FIRE_ACQUIRE_STATE = novo_fire_acquire_state()
                    else:
                        LAST_STATUS_TEXT = "rota ativa"

                    if cfg.get("send_to_arduino", True) and SERIAL_HANDLE is not None:
                        try:
                            if aim is not None and ROUTE_EXECUTOR.get("active", False):
                                pan, tilt = ponto_para_servos(aim, frame.shape, cfg)
                                enviar_servos_runtime(pan, tilt, water_on=water_on)
                            else:
                                cmd = enviar_comando_arduino(SERIAL_HANDLE, None, None, water_on=False, pressure=float(pressure_w.value))
                                atualizar_last_command(cmd, ler_serial_disponivel())
                        except Exception as e:
                            set_status(f"erro serial: {e}", "erro")

                else:
                    # Sem rota ativa: imagem 2 corre em tempo real e e usada como base da detecção.
                    fire_detection_base = aplicar_regioes_apagadas_preto(frame, HIT_MAP)
                    classes_fire = fire_class_ids if fire_class_ids else None
                    all_fire_dets = detectar_yolo(
                        fire_detection_base,
                        fire_model,
                        conf=cfg.get("fire_conf", 0.10),
                        imgsz=cfg.get("imgsz_infer", 960),
                        classes=classes_fire,
                        max_det=cfg.get("max_fire_detections", 300),
                    )
                    raw_fires = filtrar_fogos_por_nome_e_area(all_fire_dets, min_area=0)
                    raw_fires = fundir_deteccoes_fogo(raw_fires, frame.shape)

                    if raw_fires:
                        # Revisão v11:
                        # O primeiro ciclo em que fogo é detectado já congela a imagem 2,
                        # planeja a rota e aciona o jato. Não há mais espera de 1 segundo.
                        FIRE_ACQUIRE_STATE = novo_fire_acquire_state()
                        acquiring = False

                        freeze_base = aplicar_regioes_apagadas_preto(frame, HIT_MAP)
                        merged = fundir_deteccoes_fogo(raw_fires, frame.shape)
                        refined = refinar_fogos_por_cor_e_contorno(freeze_base, merged, frame.shape)
                        LAST_FIRE_DETS = refined or merged
                        current_point = ROUTE_EXECUTOR.get("impact") or (frame.shape[1] * 0.5, frame.shape[0] - 8)
                        LAST_PLAN = planejar_rota_combate(LAST_FIRE_DETS, HIT_MAP, cfg, frame.shape, current_point=current_point)

                        if LAST_PLAN.get("route"):
                            ROUTE_EXECUTOR = criar_executor_para_plan(
                                LAST_PLAN,
                                freeze_base,
                                frame.shape,
                                start_impact=ROUTE_EXECUTOR.get("impact"),
                            )
                            combat_base_frame = ROUTE_EXECUTOR["freeze_frame"]
                            panel2_frozen = True
                            LAST_STATUS_TEXT = "imagem 2 congelada no primeiro fogo; executando rota"

                            # Antes do primeiro comando, aplica segurança usando detecção atual + memória de humano.
                            safety_point0 = ponto_para_seguranca(ROUTE_EXECUTOR)
                            LAST_BLOCKED_BY_HUMAN, near_person, near_dist = humano_perto_do_impacto(
                                LAST_SAFETY_PERSON_DETS,
                                safety_point0,
                                cfg.get("human_safety_radius_px", 90),
                                usar_bbox=True,
                            )
                            if LAST_BLOCKED_BY_HUMAN:
                                ROUTE_EXECUTOR["paused_by_human"] = True
                                LAST_STATUS_TEXT = "JATO BLOQUEADO: humano em memoria/tempo real perto do impacto"

                            # Aciona imediatamente o primeiro comando de mira quando possível e seguro.
                            if cfg.get("send_to_arduino", True) and SERIAL_HANDLE is not None:
                                try:
                                    AIM_STATE = executor_to_aim_state(ROUTE_EXECUTOR)
                                    aim0 = AIM_STATE.get("aim")
                                    if aim0 is not None and not LAST_BLOCKED_BY_HUMAN:
                                        pan, tilt = ponto_para_servos(aim0, frame.shape, cfg)
                                        enviar_servos_runtime(pan, tilt, water_on=True)
                                    else:
                                        cmd = enviar_comando_arduino(SERIAL_HANDLE, None, None, water_on=False, pressure=float(pressure_w.value))
                                        atualizar_last_command(cmd, ler_serial_disponivel())
                                except Exception as e:
                                    set_status(f"erro serial: {e}", "erro")
                        else:
                            LAST_STATUS_TEXT = "fogo detectado, mas sem contorno/rota valida"
                    else:
                        FIRE_ACQUIRE_STATE = novo_fire_acquire_state()
                        acquiring = False
                        LAST_FIRE_DETS = []
                        LAST_PLAN = {"route": [], "meta": [], "mode": "sem_rota", "active_fires": [], "erased_fires": []}
                        LAST_STATUS_TEXT = "sem chama detectada"

                    # Sem rota ativa, deixa os servos/agua parados.
                    # Se uma rota acabou de ser criada neste ciclo, não envia comando de parada.
                    if (not ROUTE_EXECUTOR.get("active", False)) and cfg.get("send_to_arduino", True) and SERIAL_HANDLE is not None:
                        try:
                            cmd = enviar_comando_arduino(SERIAL_HANDLE, None, None, water_on=False, pressure=float(pressure_w.value))
                            atualizar_last_command(cmd, ler_serial_disponivel())
                        except Exception as e:
                            set_status(f"erro serial: {e}", "erro")

                LAST_DETECT_FRAME = FRAME_COUNTER

            # Atualizacao visual leve. A imagem 1 usa o frame mais recente do grabber.
            persons = LAST_SAFETY_PERSON_DETS or LAST_PERSON_DETS
            fires = LAST_FIRE_DETS
            plan_for_draw = ROUTE_EXECUTOR.get("plan") if ROUTE_EXECUTOR.get("active", False) else LAST_PLAN
            AIM_STATE = executor_to_aim_state(ROUTE_EXECUTOR)
            impact = AIM_STATE.get("impact")

            panel_original = desenhar_painel_original(
                frame,
                fires,
                persons,
                impact,
                cfg.get("human_safety_radius_px", 90),
                LAST_BLOCKED_BY_HUMAN,
                LAST_STATUS_TEXT,
            )
            panel_combate = desenhar_painel_combate_congelado(
                combat_base_frame,
                plan_for_draw,
                AIM_STATE,
                persons,
                HIT_MAP,
                cfg,
                blocked=LAST_BLOCKED_BY_HUMAN,
                frozen=panel2_frozen,
                acquiring=acquiring,
            )

            if ACTIVE_CALIB_POINT is not None:
                panel_original = desenhar_marcadores_calibracao(panel_original)
                panel_combate = desenhar_marcadores_calibracao(panel_combate)

            # JPEG um pouco mais leve para reduzir atraso de widget.
            panel1.value = bgr_to_jpeg_bytes(cv2.resize(panel_original, (760, 428), interpolation=cv2.INTER_AREA), quality=72)
            panel2.value = bgr_to_jpeg_bytes(cv2.resize(panel_combate, (760, 428), interpolation=cv2.INTER_AREA), quality=72)
            await asyncio.sleep(0.001)

    finally:
        try:
            grabber.stop()
        except Exception:
            try:
                cap.release()
            except Exception:
                pass
        try:
            if SERIAL_HANDLE is not None:
                cmd = enviar_comando_arduino(SERIAL_HANDLE, None, None, water_on=False, pressure=float(pressure_w.value))
                atualizar_last_command(cmd, ler_serial_disponivel())
        except Exception:
            pass
        set_status("parado", "info")

def iniciar_click(_=None):
    global MONITOR_RUNNING, MONITOR_TASK
    if MONITOR_RUNNING:
        set_status("ja esta executando", "warn")
        return
    MONITOR_RUNNING = True
    try:
        loop = asyncio.get_event_loop()
        MONITOR_TASK = loop.create_task(monitor_loop())
    except RuntimeError:
        MONITOR_TASK = asyncio.ensure_future(monitor_loop())


def parar_click(_=None):
    global MONITOR_RUNNING
    MONITOR_RUNNING = False
    set_status("parando...", "warn")

# ------------------------------------------------------------
# Eventos
# ------------------------------------------------------------
btn_start.on_click(iniciar_click)
btn_stop.on_click(parar_click)
btn_reset_erase.on_click(resetar_regioes_apagadas)
btn_save_setup.on_click(salvar_setup_click)
btn_load_setup.on_click(carregar_setup_click)
btn_refresh_setups.on_click(lambda _: atualizar_lista_setups())
btn_list_cameras.on_click(lambda _: atualizar_dropdown_cameras(mostrar_saida=True))
btn_refresh_videos.on_click(lambda _: atualizar_dropdown_videos(mostrar_saida=True))
btn_open_sketch.on_click(abrir_sketch_click)
btn_list_ports.on_click(lambda _: atualizar_dropdown_portas(mostrar_saida=True))
btn_connect_serial.on_click(conectar_serial_click)
btn_disconnect_serial.on_click(desconectar_serial_click)
btn_center_servos.on_click(centralizar_servos_click)
btn_ping_arduino.on_click(ping_arduino_click)
btn_apply_calib.on_click(aplicar_calibracao_click)
btn_disable_calib.on_click(desativar_calibracao_click)
btn_save_calib.on_click(salvar_calibracao_click)
btn_load_calib.on_click(carregar_calibracao_click)

for key in ["tl", "tr", "bl"]:
    calib_widgets[key]["go"].on_click(lambda _, k=key: set_active_calib_point(k))
    calib_widgets[key]["pan"].observe(lambda change, k=key: slider_calib_changed(change, k), names="value")
    calib_widgets[key]["tilt"].observe(lambda change, k=key: slider_calib_changed(change, k), names="value")

# ------------------------------------------------------------
# Interface compacta
# ------------------------------------------------------------
atualizar_paineis_vazios()
atualizar_dropdown_cameras(mostrar_saida=False)
atualizar_dropdown_videos(mostrar_saida=False)
atualizar_dropdown_portas(mostrar_saida=False)
atualizar_status_calibracao()

aba_fonte = widgets.VBox([
    widgets.HTML("<b>Fonte de vídeo</b>"),
    widgets.HBox([source_mode_w, cap_width_w, cap_height_w, cap_fps_w], layout=FULL),
    widgets.HBox([btn_list_cameras, camera_selector_w], layout=FULL),
    widgets.HBox([btn_refresh_videos, video_selector_w, video_loop_w], layout=FULL),
    widgets.HTML(f"Pasta para vídeos: <code>{VIDEO_INPUT_DIR}</code>"),
], layout=FULL)

aba_deteccao = widgets.VBox([
    param_row([param_card(fire_conf_w, "fire_conf"), param_card(person_conf_w, "person_conf"), param_card(imgsz_infer_w, "imgsz_infer")]),
    param_row([param_card(detect_every_w, "detect_every_n_frames")]),
], layout=FULL)

aba_jato = widgets.VBox([
    widgets.HTML("<b>Estratégia automática:</b> no primeiro frame com fogo detectado, congela a imagem 2, planeja sobre contornos vermelho/laranja/amarelo dentro das bboxes unificadas e aciona o jato."),
    param_row([param_card(large_area_w, "large_region_area_px"), param_card(lower_offset_w, "lower_edge_offset_px"), param_card(route_step_w, "route_step_px")]),
    param_row([param_card(route_speed_w, "route_speed_px_frame"), param_card(transition_speed_w, "transition_speed_px_frame"), param_card(jet_drop_w, "jet_drop_px")]),
    param_row([param_card(pressure_w, "jet_pressure_calib"), param_card(erase_radius_w, "erase_radius_px"), param_card(erased_fraction_w, "erased_fraction_threshold")]),
    param_row([param_card(human_radius_w, "human_safety_radius_px")]),
], layout=FULL)

aba_arduino = widgets.VBox([
    widgets.HTML("<b>Arduino</b> — envio aos servos e autorização do jato ficam ativos por padrão. O jato é bloqueado automaticamente se houver humano perto do impacto."),
    widgets.HBox([btn_open_sketch, btn_list_ports, port_selector_w], layout=FULL),
    widgets.HBox([arduino_port_w, arduino_baud_w, btn_connect_serial, btn_disconnect_serial], layout=FULL),
    widgets.HBox([btn_center_servos, btn_ping_arduino], layout=FULL),
    last_command_w,
], layout=FULL)

calib_rows = []
for key in ["tl", "tr", "bl"]:
    calib_rows.append(widgets.HBox([
        widgets.HTML(f"<b>{CALIB_LABELS[key]}</b>", layout=widgets.Layout(width="160px")),
        calib_widgets[key]["go"],
        calib_widgets[key]["pan"],
        calib_widgets[key]["tilt"],
    ], layout=FULL))

aba_calib = widgets.VBox([
    widgets.HTML("Ajuste cada ponto até o jato/laser real atingir o marcador mostrado na imagem. Depois clique em aplicar."),
    *calib_rows,
    widgets.HBox([btn_apply_calib, btn_disable_calib, btn_save_calib, btn_load_calib], layout=FULL),
    calib_status_w,
], layout=FULL)

aba_modelos_setup = widgets.VBox([
    widgets.HTML("<b>Modelos e setups</b>"),
    widgets.HBox([setup_name_w, setup_dropdown_w, btn_save_setup, btn_load_setup, btn_refresh_setups], layout=FULL),
    widgets.HBox([fire_model_path_w], layout=FULL),
    widgets.HBox([person_model_path_w], layout=FULL),
], layout=FULL)

tabs = widgets.Tab(children=[aba_fonte, aba_deteccao, aba_jato, aba_arduino, aba_calib, aba_modelos_setup], layout=FULL)
for i, title in enumerate(["Fonte", "Detecção", "Jato", "Arduino", "Calibração", "Setup/modelos"]):
    tabs.set_title(i, title)

controles = widgets.VBox([
    widgets.HTML("<h3>Monitor RGB monocular — rota por eixo da chama + memoria de humano + baixa latencia</h3>"),
    widgets.HBox([btn_start, btn_stop, btn_reset_erase, status_w], layout=FULL),
    tabs,
    out_runtime,
], layout=FULL)

paineis = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>1) Original + detecção</b>"), panel1], layout=widgets.Layout(width="50%")),
    widgets.VBox([widgets.HTML("<b>2) Combate congelado + rota + mira/impacto</b>"), panel2], layout=widgets.Layout(width="50%")),
], layout=FULL)

barra_video = widgets.VBox([
    widgets.HTML("<b>Avanço do vídeo</b> — disponível quando a fonte selecionada for vídeo da pasta."),
    video_seek_w,
], layout=FULL)

display(controles, paineis, barra_video)

[ WARN:0@9936.397] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@9936.397] global cap.cpp:478 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index
[ WARN:0@9936.397] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[ WARN:0@9936.397] global cap.cpp:478 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index
[ WARN:0@9936.397] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video2): can't open camera by index
[ WARN:0@9936.397] global cap.cpp:478 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index
[ WARN:0@9936.397] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video3): can't open camera by index
[ WARN:0@9936.397] global cap.cpp:478 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index
[ WARN:0@9936.999] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video5): can't open cam